In [25]:
import os
import random
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Tuple, List, Optional
import warnings
from itertools import combinations as _combinations
from collections import Counter

from xgboost import XGBRegressor  

from IPython.display import display

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import r2_score

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

SEED = 19537
EXTRA_SEEDS = [19537, 1584678, 17052356]  
SEEDS_TO_RUN = EXTRA_SEEDS

def seed_suffix(seed: int) -> str:
    return f"_seed{seed}"

def set_seeds(seed: int) -> None:
    global SEED, RNG, XGB_PARAMS
    SEED = int(seed)
    random.seed(SEED)
    np.random.seed(SEED)
    RNG = np.random.default_rng(SEED)
    XGB_PARAMS["random_state"] = SEED


os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings(
    "ignore",
    message=".*Falling back to prediction using DMatrix due to mismatched devices.*",
    category=UserWarning,
)

In [26]:
# Paths and configuration

NOTEBOOK_SUBDIR = "notebook 3a"

ARTIFACTS = Path("artifacts")
IN_CLEAN = ARTIFACTS / "cleaned" / "notebook 2"
IN_T2 = ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection"

OUT_REPORTS = ARTIFACTS / "reports" / NOTEBOOK_SUBDIR
OUT_META = ARTIFACTS / "metadata" / NOTEBOOK_SUBDIR

for d in [OUT_REPORTS, OUT_META]:
    d.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(SEED)

# Benchmark settings 
PRIMARY_TARGET = "lfc"
N_DRUGS_TOP_BY_COVERAGE = 100      
MIN_MEASURED_CELLS_PER_DRUG = 120   # skip drugs that become too small after arm filtering
N_SPLITS_DESIRED = 10

# Preprocessing (leakage-safe: everything fit on train fold cell lines only)
USE_PCA = True
PCA_COMPONENTS = 200

# Models
RIDGE_ALPHA = 1.0

# Elastic Net (strong linear baseline with correlated features)
USE_ELASTICNET = True
EN_ALPHA = 0.05
EN_L1_RATIO = 0.2

# Forests (tree ensembles)
USE_EXTRATREES = True
ET_N_ESTIMATORS = 400
ET_MAX_DEPTH = None
ET_MIN_SAMPLES_LEAF = 2

USE_RANDOMFOREST = True  
RF_N_ESTIMATORS = 400
RF_MAX_DEPTH = None
RF_MIN_SAMPLES_LEAF = 2

# XGB
USE_XGB = True

USE_GPU_FOR_XGB = True     # set False to force CPU
XGB_DEVICE = "cuda"        # or "cuda:0"

XGB_PARAMS = dict(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,

    random_state=SEED,

    tree_method="hist",
    device=(XGB_DEVICE if USE_GPU_FOR_XGB else "cpu"),

    n_jobs=-1,
)

# Progress logging (stdout)
PRINT_EVERY_N_DRUGS = 5      # prints one progress line every N drugs per arm
PRINT_PER_FOLD = False       # True prints per-fold sizes (can be noisy)
PRINT_SKIPS = True           # True prints first few skip reasons per arm
MAX_SKIP_PRINTS = 5     
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_EVERY_N_DRUGS = 5  # save state every N processed drugs per arm
def set_paths_for_seed(seed: int) -> dict:
    suf = seed_suffix(seed)
    paths = {
        "checkpoint_rows": OUT_REPORTS / f"prot_backbone_bench_perdrug__checkpoint_lfc{suf}.csv",
        "checkpoint_state": OUT_META / f"prot_backbone_bench_state_lfc{suf}.json",
        "detail": OUT_REPORTS / f"prot_backbone_bench_perdrug_lfc{suf}.csv",
        "perdrug_agg": OUT_REPORTS / f"prot_backbone_bench_perdrug_aggregated_lfc{suf}.csv",
        "metrics": OUT_REPORTS / f"prot_backbone_bench_metrics_lfc{suf}.csv",
    }
    return paths

In [27]:
# Helpers

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def fingerprint(path: Path) -> dict:
    st = path.stat()
    return {
        "path": str(path.resolve()),
        "size_bytes": int(st.st_size),
        "mtime_utc": datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat(),
        "sha256": sha256_file(path),
    }

def write_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def read_json_if_exists(path: Path) -> Optional[dict]:
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

BENCH_COLUMNS = [
    "seed",
    "arm", "compound_id", "fold", "model", "feature_set",
    "n_train", "n_test", "spearman", "r2",
]

def make_run_signature(selected_drugs: List[str], feature_combos: Dict[str, Tuple[str, ...]]) -> str:
    payload = {
        "PRIMARY_TARGET": PRIMARY_TARGET,
        "N_DRUGS_TOP_BY_COVERAGE": N_DRUGS_TOP_BY_COVERAGE,
        "MIN_MEASURED_CELLS_PER_DRUG": MIN_MEASURED_CELLS_PER_DRUG,
        "N_SPLITS_DESIRED": N_SPLITS_DESIRED,
        "USE_PCA": USE_PCA,
        "PCA_COMPONENTS": PCA_COMPONENTS,
        "RIDGE_ALPHA": RIDGE_ALPHA,
        "USE_ELASTICNET": USE_ELASTICNET,
        "EN_ALPHA": EN_ALPHA,
        "EN_L1_RATIO": EN_L1_RATIO,
        "USE_EXTRATREES": USE_EXTRATREES,
        "ET_N_ESTIMATORS": ET_N_ESTIMATORS,
        "ET_MAX_DEPTH": ET_MAX_DEPTH,
        "ET_MIN_SAMPLES_LEAF": ET_MIN_SAMPLES_LEAF,
        "USE_RANDOMFOREST": USE_RANDOMFOREST,
        "RF_N_ESTIMATORS": RF_N_ESTIMATORS,
        "RF_MAX_DEPTH": RF_MAX_DEPTH,
        "RF_MIN_SAMPLES_LEAF": RF_MIN_SAMPLES_LEAF,
        "USE_XGB": USE_XGB,
        "USE_GPU_FOR_XGB": USE_GPU_FOR_XGB,
        "XGB_DEVICE": XGB_DEVICE,
        "XGB_PARAMS": XGB_PARAMS,
        "FEATURE_COMBOS": {k: list(v) for k, v in feature_combos.items()},
        "selected_drugs": list(selected_drugs),
    }
    s = json.dumps(payload, sort_keys=True).encode("utf-8")
    return hashlib.sha256(s).hexdigest()[:16]

def append_rows_csv(rows: List[dict], path: Path) -> None:
    if not rows:
        return
    df = pd.DataFrame(rows)
    # enforce stable column order
    for c in BENCH_COLUMNS:
        if c not in df.columns:
            df[c] = np.nan
    df = df[BENCH_COLUMNS]
    header = not path.exists()
    df.to_csv(path, mode="a", index=False, header=header)

def load_checkpoint(run_signature: str) -> Tuple[List[dict], Dict[str, set], int]:
    bench_rows: List[dict] = []
    processed_by_arm: Dict[str, set] = {}
    saved_len = 0

    if not RESUME_FROM_CHECKPOINT:
        return bench_rows, processed_by_arm, saved_len

    state = read_json_if_exists(CHECKPOINT_STATE_JSON)
    if state is None:
        return bench_rows, processed_by_arm, saved_len

    if state.get("run_signature") != run_signature:
        print("[checkpoint] Signature mismatch, ignoring existing checkpoint.")
        return bench_rows, processed_by_arm, saved_len

    if CHECKPOINT_ROWS_CSV.exists():
        df = pd.read_csv(CHECKPOINT_ROWS_CSV)
        bench_rows = df.to_dict(orient="records")
        saved_len = len(bench_rows)

    raw = state.get("processed_by_arm", {})
    processed_by_arm = {arm: set(drugs) for arm, drugs in raw.items()}

    print(f"[checkpoint] Resumed: rows={saved_len}, arms_with_progress={len(processed_by_arm)}")
    return bench_rows, processed_by_arm, saved_len

def save_checkpoint(run_signature: str, bench_rows: List[dict], processed_by_arm: Dict[str, set], saved_len_box: dict) -> None:
    new_rows = bench_rows[saved_len_box["saved_len"]:]
    append_rows_csv(new_rows, CHECKPOINT_ROWS_CSV)
    saved_len_box["saved_len"] = len(bench_rows)

    state = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "run_signature": run_signature,
        "saved_rows": int(saved_len_box["saved_len"]),
        "processed_by_arm": {arm: sorted(list(s)) for arm, s in processed_by_arm.items()},
    }
    write_json(state, CHECKPOINT_STATE_JSON)
    if new_rows:
        print(f"[checkpoint] Wrote +{len(new_rows)} rows (total={saved_len_box['saved_len']})")
    else:
        print(f"[checkpoint] State saved (total_rows={saved_len_box['saved_len']})")

def read_parquet_strict(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet: {path}")
    return pd.read_parquet(path)

def normalise_str_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.index = df.index.astype(str)
    return df

def spearman_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.size < 2:
        return np.nan
    # rank transform with average ties
    rt = pd.Series(y_true).rank(method="average").to_numpy(dtype=float)
    rp = pd.Series(y_pred).rank(method="average").to_numpy(dtype=float)
    if np.std(rt) == 0 or np.std(rp) == 0:
        return np.nan
    return float(np.corrcoef(rt, rp)[0, 1])

def pick_group_column(cell_index: pd.DataFrame) -> str:
    candidates = ["lineage_1", "primary_disease", "lineage", "lineage_2"]
    for c in candidates:
        if c in cell_index.columns:
            return c
    return "depmap_id"

def safe_group_splits(
    cells: List[str],
    groups: pd.Series,
    n_splits_desired: int
) -> Tuple[List[Tuple[np.ndarray, np.ndarray]], str]:
    """
    Returns list of (train_idx, test_idx) splits over 'cells' indices.
    Falls back to KFold if GroupKFold is not feasible.
    """
    groups = groups.reindex(cells)
    groups = groups.fillna("Unknown").astype(str)

    n_groups = groups.nunique()
    n_cells = len(cells)
    n_splits = min(n_splits_desired, n_groups, n_cells)

    if n_splits >= 2 and n_groups >= 2:
        splitter = GroupKFold(n_splits=n_splits)
        splits = list(splitter.split(X=np.zeros((n_cells, 1)), y=np.zeros(n_cells), groups=groups.values))
        return splits, f"GroupKFold(n_splits={n_splits})"
    # fallback
    n_splits = min(max(2, min(n_splits_desired, n_cells)), n_cells)
    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    splits = list(splitter.split(np.zeros((n_cells, 1))))
    return splits, f"KFold(n_splits={n_splits}, shuffle=True, random_state={SEED})"

def has_any_observation(df: pd.DataFrame) -> pd.Series:
    if df.shape[1] == 0:
        return pd.Series(False, index=df.index)
    return df.notna().any(axis=1)

class FoldTransformer:
    """
    Imputer + scaler + optional PCA, fitted on train-fold cell lines only (leakage-safe),
    then used to transform any set of cell lines.
    """
    def __init__(self, use_pca: bool, n_components: int, random_state: int = 0):
        self.use_pca = bool(use_pca)
        self.n_components = int(n_components)
        self.random_state = int(random_state)

        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler(with_mean=True, with_std=True)
        self.pca = None

        self.keep_mask: Optional[np.ndarray] = None  # bool mask over input columns
        self.n_dropped_all_missing: int = 0

    def fit(self, X_train: np.ndarray) -> "FoldTransformer":
        X_train = np.asarray(X_train, dtype=float)

        if X_train.ndim != 2 or X_train.shape[1] == 0:
            self.keep_mask = np.zeros((0,), dtype=bool)
            self.n_dropped_all_missing = 0
            self.pca = None
            return self

        # keep columns with at least one finite observation in TRAIN
        keep = np.isfinite(X_train).any(axis=0)
        self.keep_mask = keep.astype(bool)
        self.n_dropped_all_missing = int((~self.keep_mask).sum())

        # if everything is missing in this fold, output empty embeddings for this modality
        if int(self.keep_mask.sum()) == 0:
            self.pca = None
            return self

        Xk = X_train[:, self.keep_mask]
        X_imp = self.imputer.fit_transform(Xk)
        X_std = self.scaler.fit_transform(X_imp)

        if self.use_pca:
            n = X_std.shape[0]
            d = X_std.shape[1]
            n_comp = min(self.n_components, max(1, n - 1), d)
            self.pca = PCA(n_components=n_comp, random_state=self.random_state)
            self.pca.fit(X_std)

        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)

        if self.keep_mask is None or int(self.keep_mask.sum()) == 0:
            return np.zeros((X.shape[0], 0), dtype=np.float32)

        Xk = X[:, self.keep_mask]
        X_imp = self.imputer.transform(Xk)
        X_std = self.scaler.transform(X_imp)

        if self.pca is not None:
            X_std = self.pca.transform(X_std)

        return X_std.astype(np.float32, copy=False)

def try_make_xgb():
    if not USE_XGB:
        return None, "disabled"
    try:
        try:
            mdl = XGBRegressor(**XGB_PARAMS)
            name = f"xgboost.XGBRegressor(device={XGB_PARAMS.get('device')}, tree_method={XGB_PARAMS.get('tree_method')})"
            return mdl, name
        except TypeError:
            params = dict(XGB_PARAMS)
            params.pop("device", None)
            params["tree_method"] = "gpu_hist" if USE_GPU_FOR_XGB else "hist"
            if USE_GPU_FOR_XGB:
                params["predictor"] = "gpu_predictor"
            mdl = XGBRegressor(**params)
            name = f"xgboost.XGBRegressor(tree_method={params['tree_method']})"
            return mdl, name

    except Exception:
        try:
            return HistGradientBoostingRegressor(random_state=SEED), "sklearn.HistGradientBoostingRegressor (fallback)"
        except Exception:
            return None, "unavailable"
        
def make_models():
    models = []
    models.append(("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=SEED)))

    if USE_ELASTICNET:
        models.append((
            "elasticnet",
            ElasticNet(alpha=EN_ALPHA, l1_ratio=EN_L1_RATIO, random_state=SEED, max_iter=10000)
        ))

    if USE_EXTRATREES:
        models.append((
            "extratrees",
            ExtraTreesRegressor(
                n_estimators=ET_N_ESTIMATORS,
                random_state=SEED,
                n_jobs=-1,
                max_depth=ET_MAX_DEPTH,
                min_samples_leaf=ET_MIN_SAMPLES_LEAF,
            )
        ))

    if USE_RANDOMFOREST:
        models.append((
            "randomforest",
            RandomForestRegressor(
                n_estimators=RF_N_ESTIMATORS,
                random_state=SEED,
                n_jobs=-1,
                max_depth=RF_MAX_DEPTH,
                min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            )
        ))

    return models

In [28]:
# Load cleaned backbone (Notebook 2) and arm matrices (Notebook 1 Track 2)

cell_index = read_parquet_strict(IN_CLEAN / "cell_index.parquet")
prism_long = read_parquet_strict(IN_CLEAN / "prism_long.parquet")

# Track 2 backbone matrices
rna = normalise_str_index(read_parquet_strict(IN_T2 / "rna.parquet"))
cnv = normalise_str_index(read_parquet_strict(IN_T2 / "cnv.parquet"))
mut = normalise_str_index(read_parquet_strict(IN_T2 / "mut.parquet"))

# proteomics per arm (Track 2)
prot_paths = {
    "prot_ms_ccle_gygi": IN_T2 / "prot_optional__prot_ms_ccle_gygi.parquet",
    "prot_rppa_ccle": IN_T2 / "prot_optional__prot_rppa_ccle.parquet",
    "prot_procan_depmapSanger": IN_T2 / "prot_optional__prot_procan_depmapSanger.parquet",
    "prot_combined_union": IN_T2 / "prot_optional__prot_combined_union.parquet",
}

proteomics_arms: Dict[str, pd.DataFrame] = {}
for arm, p in prot_paths.items():
    if p.exists():
        df = normalise_str_index(pd.read_parquet(p))
        proteomics_arms[arm] = df
    else:
        proteomics_arms[arm] = pd.DataFrame(index=pd.Index([], name="depmap_id"))

# Normalise cell_index depmap_id
if "depmap_id" not in cell_index.columns:
    raise ValueError("cell_index.parquet must include depmap_id column.")
cell_index["depmap_id"] = cell_index["depmap_id"].astype(str).str.strip()

group_col = pick_group_column(cell_index)
groups_all = (
    cell_index.set_index("depmap_id")[group_col]
    .astype("string")
    .fillna("Unknown")
    .astype(str)
)

core_cells = sorted(set(rna.index) & set(cnv.index) & set(mut.index))
print("Loaded core Track 2 cohort:", len(core_cells))
print("Group column for CV:", group_col)

# PRISM cohorts
prism_long["depmap_id"] = prism_long["depmap_id"].astype(str).str.strip()
prism_long["compound_id"] = prism_long["compound_id"].astype(str).str.strip()
prism_long["target"] = prism_long["target"].astype(str).str.strip().str.lower()

prism_lfc = prism_long[prism_long["target"] == PRIMARY_TARGET][["depmap_id", "compound_id", "y"]].copy()
cells_prism_lfc = set(prism_lfc["depmap_id"].unique().tolist())

Loaded core Track 2 cohort: 1079
Group column for CV: lineage_1


In [29]:
# Coverage + overlap table per arm

def summarise_missingness(df: pd.DataFrame) -> dict:
    if df.shape[0] == 0 or df.shape[1] == 0:
        return dict(
            overall_missing_pct=np.nan,
            col_missing_q50=np.nan,
            col_missing_q90=np.nan,
            col_missing_q99=np.nan,
            row_missing_q50=np.nan,
            row_missing_q90=np.nan,
            row_missing_q99=np.nan,
        )
    col_miss = df.isna().mean(axis=0)
    row_miss = df.isna().mean(axis=1)
    return dict(
        overall_missing_pct=float(df.isna().mean().mean() * 100.0),
        col_missing_q50=float(col_miss.quantile(0.50)),
        col_missing_q90=float(col_miss.quantile(0.90)),
        col_missing_q99=float(col_miss.quantile(0.99)),
        row_missing_q50=float(row_miss.quantile(0.50)),
        row_missing_q90=float(row_miss.quantile(0.90)),
        row_missing_q99=float(row_miss.quantile(0.99)),
    )

def union_platform_availability_diag(union_df: pd.DataFrame) -> Tuple[pd.DataFrame, dict]:
    """
    Diagnostics for Arm D (namespaced union):
    - which platform blocks are present per cell line
    - approximate how much missingness is due to entire platform absence
    """
    if union_df.shape[0] == 0 or union_df.shape[1] == 0:
        return pd.DataFrame(), {}

    prefixes = ["ms__", "rppa__", "procan__"]
    blocks = {pref: [c for c in union_df.columns if str(c).startswith(pref)] for pref in prefixes}

    # block present if any non-missing values in that block
    present = pd.DataFrame(index=union_df.index)
    for pref, cols in blocks.items():
        if cols:
            present[pref[:-2]] = union_df[cols].notna().any(axis=1).astype(np.int8)
        else:
            present[pref[:-2]] = 0

    # pattern label per cell line
    def patt(row) -> str:
        inc = [k for k in ["ms", "rppa", "procan"] if int(row.get(k, 0)) == 1]
        return "+".join(inc) if inc else "none"

    patterns = present.apply(patt, axis=1).rename("pattern")
    pat_counts = patterns.value_counts().rename_axis("pattern").reset_index(name="n_cells")
    pat_counts["frac_cells"] = pat_counts["n_cells"] / float(union_df.shape[0])

    # missingness contribution from full-block absence
    contrib = {}
    for pref, cols in blocks.items():
        if not cols:
            continue
        key = pref[:-2]  # ms, rppa, procan
        absent_mask = present[key].eq(0).to_numpy()
        n_absent = int(absent_mask.sum())
        n_cols = len(cols)
        # entries are fully missing when platform absent by construction
        missing_from_absence = n_absent * n_cols
        missing_total = int(union_df[cols].isna().sum().sum())
        contrib[f"{key}_frac_cells_present"] = float(present[key].mean())
        contrib[f"{key}_missing_absence_contrib"] = float(missing_from_absence / missing_total) if missing_total > 0 else np.nan

    return pat_counts, contrib

coverage_rows = []
union_patterns_df = None
union_platform_stats = {}

for arm, prot in proteomics_arms.items():
    prot = prot.copy()
    prot.index = prot.index.astype(str).str.strip()

    # only consider rows that are in the core cohort
    prot_core = prot.reindex(core_cells)

    # cell lines with any proteomics observation (arm-specific)
    has_prot = has_any_observation(prot_core)
    cells_with_prot = set(has_prot[has_prot].index.tolist())

    row = {
    "arm": arm,
    "n_cells_total_in_core": int(prot_core.shape[0]),
    "n_features": int(prot_core.shape[1]),
    "n_cells_with_any_prot": int(len(cells_with_prot)),
    "n_overlap_with_prism_lfc_cells": int(len(cells_with_prot & cells_prism_lfc)),
    }

    miss_stats = summarise_missingness(prot_core.loc[sorted(cells_with_prot)] if len(cells_with_prot) else prot_core)
    row.update(miss_stats)

    if arm == "prot_combined_union":
        union_patterns_df, union_platform_stats = union_platform_availability_diag(prot_core.loc[sorted(cells_with_prot)])
        row.update(union_platform_stats)

    coverage_rows.append(row)

prot_backbone_coverage = pd.DataFrame(coverage_rows).sort_values("n_overlap_with_prism_lfc_cells", ascending=False)
coverage_path = OUT_REPORTS / "prot_backbone_coverage_lfc.csv"
prot_backbone_coverage.to_csv(coverage_path, index=False)
print("Wrote:", coverage_path)

if union_patterns_df is not None and union_patterns_df.shape[0] > 0:
    union_patterns_path = OUT_REPORTS / "prot_combined_union__platform_availability_patterns_lfc.csv"
    union_patterns_df.to_csv(union_patterns_path, index=False)
    print("Wrote:", union_patterns_path)

Wrote: artifacts\reports\notebook 3a\prot_backbone_coverage_lfc.csv
Wrote: artifacts\reports\notebook 3a\prot_combined_union__platform_availability_patterns_lfc.csv


In [30]:
# Quick benchmark: Ridge + XGB on backbone vs backbone+PROT per arm

# Select drugs by coverage in PRISM LFC (not by performance)
drug_cov = prism_lfc.groupby("compound_id")["depmap_id"].nunique().sort_values(ascending=False)
selected_drugs = drug_cov.head(N_DRUGS_TOP_BY_COVERAGE).index.tolist()
print(f"Selected top-{len(selected_drugs)} drugs by PRISM LFC coverage.")

# Pull only the relevant pairs for the selected drugs to keep memory and grouping fast
prism_sel = prism_lfc[prism_lfc["compound_id"].isin(selected_drugs)].copy()
drug_to_pairs = {k: v for k, v in prism_sel.groupby("compound_id", sort=False)}

xgb_model, xgb_name = try_make_xgb()
print("Non-linear model:", xgb_name)

MODALITIES = ("rna", "cnv", "mut", "prot")

FEATURE_COMBOS: Dict[str, Tuple[str, ...]] = {}
for r in range(1, len(MODALITIES) + 1):
    for combo in _combinations(MODALITIES, r):
        FEATURE_COMBOS["+".join(combo)] = combo

def fit_fold_embeddings(
    eligible_cells: List[str],
    train_cells: List[str],
    arm: str,
    prot_df: pd.DataFrame,
) -> Tuple[Dict[str, np.ndarray], Dict[str, int]]:
    """
    Fit per-modality transformers on train_cells only, transform all eligible_cells.
    Returns:
      mats: dict with keys: rna, cnv, mut, prot (each is [n_cells, d])
      info: dims and dropped-all-missing counts
    """
    rna_tr = FoldTransformer(USE_PCA, PCA_COMPONENTS, random_state=SEED + 0).fit(rna.loc[train_cells].to_numpy())
    cnv_tr = FoldTransformer(USE_PCA, PCA_COMPONENTS, random_state=SEED + 1).fit(cnv.loc[train_cells].to_numpy())
    mut_tr = FoldTransformer(USE_PCA, PCA_COMPONENTS, random_state=SEED + 2).fit(mut.loc[train_cells].to_numpy())

    Xr = rna_tr.transform(rna.loc[eligible_cells].to_numpy())
    Xc = cnv_tr.transform(cnv.loc[eligible_cells].to_numpy())
    Xm = mut_tr.transform(mut.loc[eligible_cells].to_numpy())

    Xp = np.zeros((len(eligible_cells), 0), dtype=np.float32)
    dropped_all_missing_prot = 0
    if prot_df is not None and prot_df.shape[1] > 0:
        prot_tr = FoldTransformer(USE_PCA, PCA_COMPONENTS, random_state=SEED + 3).fit(
            prot_df.loc[train_cells].to_numpy()
        )
        Xp = prot_tr.transform(prot_df.loc[eligible_cells].to_numpy())
        dropped_all_missing_prot = int(getattr(prot_tr, "n_dropped_all_missing", 0))

    mats = {"rna": Xr, "cnv": Xc, "mut": Xm, "prot": Xp}

    info = {
        "d_rna": int(Xr.shape[1]),
        "d_cnv": int(Xc.shape[1]),
        "d_mut": int(Xm.shape[1]),
        "d_prot": int(Xp.shape[1]),
        "dropped_all_missing_rna": int(getattr(rna_tr, "n_dropped_all_missing", 0)),
        "dropped_all_missing_cnv": int(getattr(cnv_tr, "n_dropped_all_missing", 0)),
        "dropped_all_missing_mut": int(getattr(mut_tr, "n_dropped_all_missing", 0)),
        "dropped_all_missing_prot": int(dropped_all_missing_prot),
    }
    return mats, info

for run_seed in SEEDS_TO_RUN:
    set_seeds(run_seed)
    p = set_paths_for_seed(run_seed)
    suf = seed_suffix(run_seed)
    # point the existing checkpoint helpers at the seed-specific files
    CHECKPOINT_ROWS_CSV = p["checkpoint_rows"]
    CHECKPOINT_STATE_JSON = p["checkpoint_state"]

    # skip: if this seed already finished, do nothing
    if p["detail"].exists() and p["perdrug_agg"].exists() and p["metrics"].exists():
        print(f"[seed={run_seed}] Outputs already exist, skipping.")
        continue

    print(f"\n=== Running LFC seed {run_seed} ===")

    # Checkpoint load (resume)
    RUN_SIGNATURE = make_run_signature(selected_drugs=selected_drugs, feature_combos=FEATURE_COMBOS)
    bench_rows, processed_by_arm, _saved_len = load_checkpoint(RUN_SIGNATURE)
    _ckpt = {"saved_len": int(_saved_len)}

    for r in bench_rows:
        if ("seed" not in r) or pd.isna(r.get("seed")):
            r["seed"] = int(SEED)

    for arm, prot in proteomics_arms.items():
        if prot.shape[0] == 0 or prot.shape[1] == 0:
            print(f"[WARN] Skipping benchmark for {arm} (no proteomics loaded).")
            continue
        
        # Resume support per arm
        processed_drugs = processed_by_arm.get(arm, set())
        processed_by_arm[arm] = processed_drugs
        processed_this_run = 0

        prot = prot.copy()
        prot.index = prot.index.astype(str).str.strip()
        prot_core = prot.reindex(core_cells)

        # Define eligible cells for this arm as those with any proteomics values (so base vs base+prot is comparable)
        has_prot = has_any_observation(prot_core)
        eligible_cells = sorted(has_prot[has_prot].index.tolist())

        if len(eligible_cells) < 200:
            print(f"[WARN] {arm}: too few eligible cells ({len(eligible_cells)}), skipping.")
            continue

        # Precompute CV splits once per arm (over eligible cells)
        arm_groups = groups_all.reindex(eligible_cells).fillna("Unknown").astype(str)
        splits, split_name = safe_group_splits(eligible_cells, arm_groups, N_SPLITS_DESIRED)
        print(f"{arm}: CV={split_name}, eligible_cells={len(eligible_cells)}")

        # Build a fold map: cell_id -> fold index (test fold)
        fold_map = {}
        for fold_i, (_, test_idx) in enumerate(splits):
            for j in test_idx:
                fold_map[eligible_cells[int(j)]] = fold_i

        # Pre-fit fold-specific embeddings (transformers are fitted without labels, so re-usable across drugs)
        fold_cache = {}
        for fold_i, (train_idx, test_idx) in enumerate(splits):
            train_cells = [eligible_cells[int(j)] for j in train_idx]
            # Restrict prot_core to eligible cells, and align
            prot_elig = prot_core.loc[eligible_cells]
            mats, info = fit_fold_embeddings(
                eligible_cells=eligible_cells,
                train_cells=train_cells,
                arm=arm,
                prot_df=prot_elig,
            )

            cell_to_row = {cid: i for i, cid in enumerate(eligible_cells)}

            fold_cache[fold_i] = {
                "mats": mats,
                "cell_to_row": cell_to_row,
                "info": info,
                "train_cells_set": set(train_cells),
            }

        n_selected = len(selected_drugs)
        n_seen = 0
        n_evaluated = 0
        n_skipped_missing_pairs = 0
        n_skipped_low_cells = 0
        n_skipped_no_valid_folds = 0
        skip_prints_used = 0

        # Evaluate per drug
        for drug_i, drug in enumerate(selected_drugs, start=1):
            if drug in processed_drugs:
                continue
            n_seen += 1

            pairs = drug_to_pairs.get(drug)
            if pairs is None or pairs.shape[0] == 0:
                n_skipped_missing_pairs += 1
                if PRINT_SKIPS and (skip_prints_used < MAX_SKIP_PRINTS):
                    print(f"[{arm}] ({drug_i}/{n_selected}) drug={drug}: no PRISM pairs found, skipping")
                    skip_prints_used += 1

                processed_drugs.add(drug)
                processed_this_run += 1
                if (processed_this_run % CHECKPOINT_EVERY_N_DRUGS) == 0:
                    save_checkpoint(RUN_SIGNATURE, bench_rows, processed_by_arm, _ckpt)
                continue 

            # Restrict to eligible cells for this arm
            df = pairs[pairs["depmap_id"].isin(eligible_cells)][["depmap_id", "y"]].copy()
            n_cells_raw = int(df["depmap_id"].nunique())

            if n_cells_raw < MIN_MEASURED_CELLS_PER_DRUG:
                n_skipped_low_cells += 1
                if PRINT_SKIPS and (skip_prints_used < MAX_SKIP_PRINTS):
                    print(
                        f"[{arm}] ({drug_i}/{n_selected}) drug={drug}: "
                        f"only {n_cells_raw} cells after arm filter (<{MIN_MEASURED_CELLS_PER_DRUG}), skipping"
                    )
                    skip_prints_used += 1
                processed_drugs.add(drug)
                processed_this_run += 1
                if (processed_this_run % CHECKPOINT_EVERY_N_DRUGS) == 0:
                    save_checkpoint(RUN_SIGNATURE, bench_rows, processed_by_arm, _ckpt)
                continue
                

            # Ensure one measurement per cell line
            df = df.groupby("depmap_id", as_index=False)["y"].mean()

            cell_ids = df["depmap_id"].astype(str).tolist()
            y_all = df["y"].to_numpy(dtype=float)

            # Determine fold id for each sample (based on depmap_id)
            fold_ids = np.array([fold_map.get(cid, -1) for cid in cell_ids], dtype=int)
            valid = fold_ids >= 0
            cell_ids = [c for c, v in zip(cell_ids, valid) if v]
            y_all = y_all[valid]
            fold_ids = fold_ids[valid]

            if len(cell_ids) < MIN_MEASURED_CELLS_PER_DRUG:
                n_skipped_low_cells += 1
                if PRINT_SKIPS and (skip_prints_used < MAX_SKIP_PRINTS):
                    print(
                        f"[{arm}] ({drug_i}/{n_selected}) drug={drug}: "
                        f"{len(cell_ids)} cells after fold-map filter (<{MIN_MEASURED_CELLS_PER_DRUG}), skipping"
                    )
                    skip_prints_used += 1

                processed_drugs.add(drug)
                processed_this_run += 1
                if (processed_this_run % CHECKPOINT_EVERY_N_DRUGS) == 0:
                    save_checkpoint(RUN_SIGNATURE, bench_rows, processed_by_arm, _ckpt)
                continue

            n_evaluated += 1

            if (drug_i % PRINT_EVERY_N_DRUGS == 0) or (drug_i == 1) or (drug_i == n_selected):
                print(f"[{arm}] ({drug_i}/{n_selected}) Evaluating drug={drug} with {len(cell_ids)} cells")

            any_fold_ran = False

            for fold_i in sorted(fold_cache.keys()):
                in_test = fold_ids == fold_i
                n_test = int(in_test.sum())
                n_train = int((~in_test).sum())
                if n_test < 10 or n_train < 30:
                    continue

                any_fold_ran = True

                if PRINT_PER_FOLD:
                    print(f"    fold={fold_i}: n_train={n_train}, n_test={n_test}")

                cache = fold_cache[fold_i]
                cell_to_row = cache["cell_to_row"]
                mats = cache["mats"]

                idx_all = np.array([cell_to_row[cid] for cid in cell_ids], dtype=int)
                idx_train = idx_all[~in_test]
                idx_test = idx_all[in_test]

                y_train = y_all[~in_test]
                y_test = y_all[in_test]

                bench_rows.append({
                    "seed": int(SEED),
                    "arm": arm,
                    "compound_id": drug,
                    "fold": int(fold_i),
                    "model": "null_mean",
                    "feature_set": "none",
                    "n_train": int(len(y_train)),
                    "n_test": int(len(y_test)),
                    "spearman": 0.0,
                    "r2": float(r2_score(y_test, np.full_like(y_test, float(y_train.mean())))),
                })

                def build_X(combo_keys: Tuple[str, ...]) -> np.ndarray:
                    # Require every requested modality to be present, otherwise skip this combo
                    for k in combo_keys:
                        if mats[k].shape[1] == 0:
                            return np.zeros((len(eligible_cells), 0), dtype=np.float32)
                    return np.concatenate([mats[k] for k in combo_keys], axis=1)

                # Build feature matrices once per fold
                X_by_combo = {name: build_X(keys) for name, keys in FEATURE_COMBOS.items()}

                # Linear and forest models
                for model_name, model in make_models():
                    for feat_name, Xmat_all in X_by_combo.items():
                        if Xmat_all.shape[1] == 0:
                            continue

                        X_train = Xmat_all[idx_train]
                        X_test = Xmat_all[idx_test]

                        model.fit(X_train, y_train)
                        pred = model.predict(X_test)

                        bench_rows.append({
                            "seed": int(SEED),
                            "arm": arm,
                            "compound_id": drug,
                            "fold": int(fold_i),
                            "model": model_name,
                            "feature_set": feat_name,
                            "n_train": int(len(y_train)),
                            "n_test": int(len(y_test)),
                            "spearman": spearman_corr(y_test, pred),
                            "r2": float(r2_score(y_test, pred)),
                        })

                # XGB (recreated per fit)
                if xgb_model is not None:
                    for feat_name, Xmat_all in X_by_combo.items():
                        if Xmat_all.shape[1] == 0:
                            continue

                        X_train = Xmat_all[idx_train]
                        X_test = Xmat_all[idx_test]

                        mdl, _ = try_make_xgb()
                        if mdl is None:
                            break

                        mdl.fit(X_train, y_train)
                        global _XGB_DEVICE_PRINTED
                        if not _XGB_DEVICE_PRINTED:
                            cfg = mdl.get_booster().save_config()
                            i = cfg.find('"device"')
                            print("XGB config snippet:", cfg[i:i+120] if i != -1 else "device not found in config")
                            _XGB_DEVICE_PRINTED = True

                        pred = mdl.predict(X_test)

                        bench_rows.append({
                            "seed": int(SEED),
                            "arm": arm,
                            "compound_id": drug,
                            "fold": int(fold_i),
                            "model": "xgb_quick",
                            "feature_set": feat_name,
                            "n_train": int(len(y_train)),
                            "n_test": int(len(y_test)),
                            "spearman": spearman_corr(y_test, pred),
                            "r2": float(r2_score(y_test, pred)),
                        })


            if not any_fold_ran:
                n_skipped_no_valid_folds += 1
                if PRINT_SKIPS and (skip_prints_used < MAX_SKIP_PRINTS):
                    print(
                        f"[{arm}] ({drug_i}/{n_selected}) drug={drug}: "
                        "no folds met min sizes (test>=10 and train>=30), skipping"
                    )
                    skip_prints_used += 1

            processed_drugs.add(drug)
            processed_this_run += 1
            if (processed_this_run % CHECKPOINT_EVERY_N_DRUGS) == 0:
                save_checkpoint(RUN_SIGNATURE, bench_rows, processed_by_arm, _ckpt)

        # End-of-arm summary
        print(
            f"[{arm}] Done: seen={n_seen}, evaluated={n_evaluated}, "
            f"skipped_no_pairs={n_skipped_missing_pairs}, skipped_low_cells={n_skipped_low_cells}, "
            f"skipped_no_valid_folds={n_skipped_no_valid_folds}"
        )
        save_checkpoint(RUN_SIGNATURE, bench_rows, processed_by_arm, _ckpt)


    bench_detail = pd.DataFrame(bench_rows)
    detail_path = p["detail"]
    bench_detail.to_csv(detail_path, index=False)
    print("Wrote:", detail_path, bench_detail.shape)

    # Summarise: per arm, model, feature_set
    if bench_detail.shape[0] == 0:
        raise RuntimeError("Benchmark produced no rows. Consider lowering MIN_MEASURED_CELLS_PER_DRUG or N_DRUGS_TOP_BY_COVERAGE.")

    # Per-drug mean across folds
    drug_means = (
        bench_detail
        .groupby(["seed", "arm", "model", "feature_set", "compound_id"], as_index=False)
        .agg(
            spearman=("spearman", "mean"),
            r2=("r2", "mean"),
            n_folds=("fold", "nunique"),
            n_test_total=("n_test", "sum"),
        )
    )

    bench_summary = (
        drug_means
        .groupby(["seed", "arm", "model", "feature_set"], as_index=False)
        .agg(
            n_drugs=("compound_id", "nunique"),
            mean_spearman=("spearman", "mean"),
            median_spearman=("spearman", "median"),
            mean_r2=("r2", "mean"),
            median_r2=("r2", "median"),
            mean_folds=("n_folds", "mean"),
        )
    )

    def add_uplift(df: pd.DataFrame) -> pd.DataFrame:
        base = df[df["feature_set"] == "rna+cnv+mut"].set_index(["seed", "arm", "model"])
        full = df[df["feature_set"] == "rna+cnv+mut+prot"].set_index(["seed", "arm", "model"])
        n_base = base.shape[0]
        n_full = full.shape[0]
        if n_base == 0:
            print("[WARN] add_uplift: no rows found for feature_set='rna+cnv+mut' — delta columns will be NaN.")
        if n_full == 0:
            print("[WARN] add_uplift: no rows found for feature_set='rna+cnv+mut+prot' — delta columns will be NaN.")
        merged = full.join(base, lsuffix="_full", rsuffix="_base", how="left")
        out = merged.reset_index()
        n_nan_delta = out["mean_spearman_full"].isna().sum() + out["mean_spearman_base"].isna().sum()
        if n_nan_delta > 0:
            print(f"[WARN] add_uplift: {n_nan_delta} NaN values in uplift join — some arms may lack a baseline or full feature set.")
        out["delta_mean_spearman"] = out["mean_spearman_full"] - out["mean_spearman_base"]
        out["delta_mean_r2"] = out["mean_r2_full"] - out["mean_r2_base"]
        return out

    uplift = add_uplift(bench_summary)

    metrics_path = p["metrics"]
    uplift.to_csv(metrics_path, index=False)
    print("Wrote:", metrics_path)

    coverage_df = prot_backbone_coverage.set_index("arm")

    uplift_ridge = uplift[uplift["model"] == "ridge"].copy()
    if uplift_ridge.shape[0] > 0:
        uplift_ridge["n_overlap_with_prism_lfc_cells"] = uplift_ridge["arm"].map(
            lambda a: int(coverage_df.loc[a, "n_overlap_with_prism_lfc_cells"]) if a in coverage_df.index else 0
        )
        uplift_ridge["overall_missing_pct"] = uplift_ridge["arm"].map(
            lambda a: float(coverage_df.loc[a, "overall_missing_pct"]) if a in coverage_df.index else np.nan
        )

        # Score: uplift + small bonus for overlap, small penalty for missingness
        uplift_ridge["score"] = (
            uplift_ridge["delta_mean_spearman"].fillna(-1e9)
            + 0.0002 * uplift_ridge["n_overlap_with_prism_lfc_cells"].astype(float)
            - 0.001 * uplift_ridge["overall_missing_pct"].fillna(100.0)
        )
        uplift_ridge = uplift_ridge.sort_values("score", ascending=False)
        suggested = uplift_ridge["arm"].head(2).tolist()
    else:
        suggested = []

    drug_means_rank = drug_means.drop_duplicates(
        subset=["seed", "arm", "model", "feature_set", "compound_id", "spearman", "r2", "n_folds", "n_test_total"]
    ).copy()

    per_drug_agg_path = p["perdrug_agg"]
    drug_means_rank.to_csv(per_drug_agg_path, index=False)
    print("Wrote:", per_drug_agg_path)

    # Overall top/bottom 10 across all arms/models/feature sets
    overall_top10 = (
        drug_means_rank
        .sort_values(["spearman", "r2", "n_folds", "n_test_total"], ascending=[False, False, False, False])
        .head(10)
        .reset_index(drop=True)
    )

    overall_bottom10 = (
        drug_means_rank
        .sort_values(["spearman", "r2", "n_folds", "n_test_total"], ascending=[True, True, False, False])
        .head(10)
        .reset_index(drop=True)
    )

    overall_top10_path = OUT_REPORTS / f"prot_backbone_bench_top10_overall_lfc{suf}.csv"
    overall_bottom10_path = OUT_REPORTS / f"prot_backbone_bench_bottom10_overall_lfc{suf}.csv"
    overall_top10.to_csv(overall_top10_path, index=False)
    overall_bottom10.to_csv(overall_bottom10_path, index=False)
    print("Wrote:", overall_top10_path)
    print("Wrote:", overall_bottom10_path)

    # top/bottom within each group 
    def _top_bottom_by_group(g: pd.DataFrame, n: int = 10) -> pd.DataFrame:
        top = (
            g.sort_values(["spearman", "r2", "n_folds", "n_test_total"], ascending=[False, False, False, False])
            .head(n)
            .copy()
        )
        top["rank_block"] = "top"

        bottom = (
            g.sort_values(["spearman", "r2", "n_folds", "n_test_total"], ascending=[True, True, False, False])
            .head(n)
            .copy()
        )
        bottom["rank_block"] = "bottom"

        return pd.concat([top, bottom], ignore_index=True)

    def _top_bottom_by_group_loop(df: pd.DataFrame, group_cols: List[str], n: int = 10) -> pd.DataFrame:
        parts = []
        for _, g in df.groupby(group_cols, sort=False):
            parts.append(_top_bottom_by_group(g, n=n))
        if not parts:
            return pd.DataFrame(columns=list(df.columns) + ["rank_block"])
        return pd.concat(parts, ignore_index=True)

    # Top/bottom 10 within each (arm, model, feature_set)
    per_group_top_bottom = _top_bottom_by_group_loop(
        drug_means_rank,
        group_cols=["arm", "model", "feature_set"],
        n=10,
    )

    per_group_top_bottom_path = OUT_REPORTS / f"prot_backbone_bench_top_bottom10_by_group_lfc{suf}.csv"
    per_group_top_bottom.to_csv(per_group_top_bottom_path, index=False)
    print("Wrote:", per_group_top_bottom_path)

Selected top-100 drugs by PRISM LFC coverage.
Non-linear model: xgboost.XGBRegressor(device=cuda, tree_method=hist)
[seed=19537] Outputs already exist, skipping.
[seed=1584678] Outputs already exist, skipping.
[seed=17052356] Outputs already exist, skipping.


In [31]:
TARGET = PRIMARY_TARGET 
ALL_SEEDS = list(EXTRA_SEEDS)

def parse_seed_from_name(path: Path) -> Optional[int]:
    s = path.stem
    if "_seed" in s:
        try:
            return int(s.split("_seed")[-1])
        except Exception:
            return None
    return None

def load_perdrug_agg_for_seeds(out_dir: Path, seeds: List[int]) -> pd.DataFrame:
    """
    Loads per-drug aggregated outputs for multiple seeds into one DataFrame.
    It looks for:
      - per-seed files: set_paths_for_seed(seed)["perdrug_agg"]
      - plus the legacy base file without suffix (optional)
    """
    files = []

    for s in seeds:
        fp = set_paths_for_seed(s)["perdrug_agg"]
        if fp.exists():
            files.append(fp)
        else:
            print("Missing:", fp)

    legacy = out_dir / f"prot_backbone_bench_perdrug_aggregated_{TARGET}.csv"
    if legacy.exists():
        files.append(legacy)

    # De-dup
    files = list(dict.fromkeys(files))

    if not files:
        raise FileNotFoundError("No per-drug aggregated files found for the requested seeds.")

    dfs = []
    for fp in files:
        df = pd.read_csv(fp)

        if "seed" not in df.columns or df["seed"].isna().all():
            inferred = parse_seed_from_name(fp)
            df["seed"] = inferred

        df["seed"] = pd.to_numeric(df["seed"], errors="coerce")
        df = df.dropna(subset=["seed"]).copy()
        df["seed"] = df["seed"].astype(int)

        # Keep only requested seeds
        df = df[df["seed"].isin(seeds)].copy()

        # Numeric clean
        for c in ["spearman", "r2", "n_folds", "n_test_total"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True).drop_duplicates()
    return merged

merged = load_perdrug_agg_for_seeds(OUT_REPORTS, ALL_SEEDS)

merged_path = OUT_REPORTS / f"prot_backbone_bench_perdrug_aggregated_{TARGET}__merged_{len(ALL_SEEDS)}seeds.csv"
merged.to_csv(merged_path, index=False)
print("Wrote:", merged_path, merged.shape)

# Seed performance overall
seed_overall = (
    merged
    .groupby("seed", as_index=False)
    .agg(
        n_rows=("compound_id", "size"),
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", "mean"),
        median_spearman=("spearman", "median"),
        std_spearman=("spearman", "std"),
        mean_r2=("r2", "mean"),
        median_r2=("r2", "median"),
    )
    .sort_values(["mean_spearman", "median_spearman"], ascending=[False, False])
)

seed_overall_path = OUT_REPORTS / f"seed_overall_summary_{TARGET}.csv"
seed_overall.to_csv(seed_overall_path, index=False)
print("Wrote:", seed_overall_path)

best_seed = int(seed_overall.iloc[0]["seed"]) if seed_overall.shape[0] else None
worst_seed = int(seed_overall.iloc[-1]["seed"]) if seed_overall.shape[0] else None
print(f"Best seed overall: {best_seed}")
print(f"Worst seed overall: {worst_seed}")

display(seed_overall)

# Best and worst combo per seed
seed_combo = (
    merged
    .groupby(["seed", "arm", "model", "feature_set"], as_index=False)
    .agg(
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", "mean"),
        median_spearman=("spearman", "median"),
        std_spearman=("spearman", "std"),
        mean_r2=("r2", "mean"),
        median_r2=("r2", "median"),
    )
)

best_combo_per_seed = (
    seed_combo
    .sort_values(["seed", "mean_spearman", "mean_r2", "n_drugs"], ascending=[True, False, False, False])
    .groupby("seed", as_index=False)
    .head(1)
    .reset_index(drop=True)
    .assign(rank_block="best")
)

worst_combo_per_seed = (
    seed_combo
    .sort_values(["seed", "mean_spearman", "mean_r2", "n_drugs"], ascending=[True, True, True, False])
    .groupby("seed", as_index=False)
    .head(1)
    .reset_index(drop=True)
    .assign(rank_block="worst")
)

seed_best_worst_combo = (
    pd.concat([best_combo_per_seed, worst_combo_per_seed], ignore_index=True)
    .sort_values(["seed", "rank_block"])
)

seed_best_worst_combo_path = OUT_REPORTS / f"seed_best_worst_combo_{TARGET}.csv"
seed_best_worst_combo.to_csv(seed_best_worst_combo_path, index=False)
print("Wrote:", seed_best_worst_combo_path)

display(seed_best_worst_combo)

# Best and worst drugs across seeds
best_per_seed_drug = (
    merged
    .sort_values(
        ["seed", "compound_id", "spearman", "r2", "n_folds", "n_test_total"],
        ascending=[True, True, False, False, False, False]
    )
    .groupby(["seed", "compound_id"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

drug_across_seeds = (
    best_per_seed_drug
    .groupby("compound_id", as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        mean_best_spearman=("spearman", "mean"),
        median_best_spearman=("spearman", "median"),
        std_best_spearman=("spearman", "std"),
        mean_best_r2=("r2", "mean"),
    )
    .sort_values(["mean_best_spearman", "median_best_spearman"], ascending=[False, False])
)

def mode_combo(sub: pd.DataFrame) -> pd.Series:
    combos = list(zip(sub["arm"], sub["model"], sub["feature_set"]))
    m = Counter(combos).most_common(1)[0][0] if combos else ("", "", "")
    return pd.Series({"mode_arm": m[0], "mode_model": m[1], "mode_feature_set": m[2]})

combo_mode = best_per_seed_drug.groupby("compound_id").apply(mode_combo, include_groups=False).reset_index()
drug_across_seeds = drug_across_seeds.merge(combo_mode, on="compound_id", how="left")

drug_across_seeds_path = OUT_REPORTS / f"drug_bestcombo_across_seeds_{TARGET}.csv"
drug_across_seeds.to_csv(drug_across_seeds_path, index=False)
print("Wrote:", drug_across_seeds_path)

top_drugs = drug_across_seeds.head(15).reset_index(drop=True)
bottom_drugs = (
    drug_across_seeds
    .sort_values(["mean_best_spearman", "median_best_spearman"], ascending=[True, True])
    .head(15)
    .reset_index(drop=True)
)

top_drugs_path = OUT_REPORTS / f"top15_drugs_across_seeds_{TARGET}.csv"
bottom_drugs_path = OUT_REPORTS / f"bottom15_drugs_across_seeds_{TARGET}.csv"
top_drugs.to_csv(top_drugs_path, index=False)
bottom_drugs.to_csv(bottom_drugs_path, index=False)
print("Wrote:", top_drugs_path)
print("Wrote:", bottom_drugs_path)

print("\nTop drugs across seeds (mean of per-seed best combos):")
display(top_drugs)

print("\nBottom drugs across seeds (mean of per-seed best combos):")
display(bottom_drugs)

# Single best and worst observed rows overall
best_rows = (
    merged
    .sort_values(["spearman", "r2", "n_folds", "n_test_total"], ascending=[False, False, False, False])
    .head(20)
    .reset_index(drop=True)
)

worst_rows = (
    merged
    .sort_values(["spearman", "r2", "n_folds", "n_test_total"], ascending=[True, True, False, False])
    .head(20)
    .reset_index(drop=True)
)

best_rows_path = OUT_REPORTS / f"best20_rows_overall_{TARGET}.csv"
worst_rows_path = OUT_REPORTS / f"worst20_rows_overall_{TARGET}.csv"
best_rows.to_csv(best_rows_path, index=False)
worst_rows.to_csv(worst_rows_path, index=False)
print("Wrote:", best_rows_path)
print("Wrote:", worst_rows_path)

print("\nBest observed rows (seed + drug + combo):")
display(best_rows.head(10))

print("\nWorst observed rows (seed + drug + combo):")
display(worst_rows.head(10))


Wrote: artifacts\reports\notebook 3a\prot_backbone_bench_perdrug_aggregated_lfc__merged_3seeds.csv (90000, 9)
Wrote: artifacts\reports\notebook 3a\seed_overall_summary_lfc.csv
Best seed overall: 1584678
Worst seed overall: 19537


,seed,n_rows,n_drugs,mean_spearman,median_spearman,std_spearman,mean_r2,median_r2
1,1584678,30000,100,0.024498,0.018463,0.074801,-1.061142,-0.132591
2,17052356,30000,100,0.024297,0.018413,0.075061,-1.061331,-0.133040
0,19537,30000,100,0.024142,0.018275,0.075193,-1.061233,-0.133039


Wrote: artifacts\reports\notebook 3a\seed_best_worst_combo_lfc.csv


,seed,arm,model,feature_set,n_drugs,mean_spearman,median_spearman,std_spearman,mean_r2,median_r2,rank_block
0,19537,prot_combined_union,extratrees,rna,100,0.056127,0.041362,0.085322,-0.046337,-0.038020,best
3,19537,prot_procan_depmapSanger,xgb_quick,mut,100,-0.015648,-0.013502,0.057706,-0.156044,-0.145560,worst
1,1584678,prot_combined_union,extratrees,rna,100,0.056180,0.049660,0.081684,-0.046691,-0.038622,best
4,1584678,prot_ms_ccle_gygi,randomforest,mut,100,-0.015446,-0.018424,0.069733,-0.113339,-0.095523,worst
2,17052356,prot_combined_union,extratrees,rna,100,0.054574,0.046026,0.082627,-0.045693,-0.039498,best
5,17052356,prot_ms_ccle_gygi,xgb_quick,mut,100,-0.016068,-0.016462,0.065519,-0.183919,-0.175007,worst


Wrote: artifacts\reports\notebook 3a\drug_bestcombo_across_seeds_lfc.csv
Wrote: artifacts\reports\notebook 3a\top15_drugs_across_seeds_lfc.csv
Wrote: artifacts\reports\notebook 3a\bottom15_drugs_across_seeds_lfc.csv

Top drugs across seeds (mean of per-seed best combos):


...

,compound_id,n_seeds,mean_best_spearman,median_best_spearman,std_best_spearman,mean_best_r2,mode_arm,mode_model,mode_feature_set
0,ATRACTYLENOLIDE-I (BRD:BRD-K00003732-001-01-9),3,0.093762,0.093762,0.000000,-8.936261,prot_procan_depmapSanger,ridge,cnv+mut
1,CANGRELOR (BRD:BRD-K00004669-001-01-9),3,0.100406,0.088796,0.020109,-0.230446,prot_ms_ccle_gygi,ridge,mut+prot
2,DOTMP (BRD:BRD-K00004678-001-01-9),3,0.108781,0.108781,0.000000,-7.895068,prot_ms_ccle_gygi,ridge,rna+cnv+mut
3,KR-33494 (BRD:BRD-K00004587-001-01-9),3,0.109092,0.109092,0.000000,-1.491830,prot_ms_ccle_gygi,ridge,cnv+mut
4,RIBOFLAVIN-TETRABUTYRATE (BRD:BRD-K74227494-00...,3,0.113855,0.113855,0.000000,-0.634848,prot_ms_ccle_gygi,ridge,cnv+prot
5,RIPRETINIB (BRD:BRD-K00005241-001-01-9),3,0.115173,0.115173,0.000000,-0.294616,prot_ms_ccle_gygi,elasticnet,prot
6,NN-DNJ (BRD:BRD-K00004640-001-01-9),3,0.115700,0.112849,0.013010,-0.049645,prot_combined_union,randomforest,prot
7,E-2001 (BRD:BRD-K00004561-001-01-9),3,0.115993,0.114406,0.017419,-0.014851,prot_procan_depmapSanger,randomforest,mut
8,ASP-6537 (BRD:BRD-K82239282-001-02-9),3,0.116199,0.116199,0.000000,-0.122831,prot_ms_ccle_gygi,elasticnet,mut
9,WAY-255348 (BRD:BRD-K00004556-001-01-9),3,0.118601,0.119482,0.014727,-0.048644,prot_ms_ccle_gygi,randomforest,mut


Wrote: artifacts\reports\notebook 3a\best20_rows_overall_lfc.csv
Wrote: artifacts\reports\notebook 3a\worst20_rows_overall_lfc.csv

Best observed rows (seed + drug + combo):


,arm,model,feature_set,compound_id,spearman,r2,n_folds,n_test_total,seed
0,prot_ms_ccle_gygi,randomforest,rna+prot,ELACESTRANT (BRD:BRD-K00003295-300-01-9),0.405982,-0.070051,10,269,19537
1,prot_ms_ccle_gygi,randomforest,rna+prot,ELACESTRANT (BRD:BRD-K00003295-300-01-9),0.405176,-0.080686,10,269,17052356
2,prot_ms_ccle_gygi,extratrees,rna+prot,ELACESTRANT (BRD:BRD-K00003295-300-01-9),0.402112,-0.093305,10,269,17052356
3,prot_ms_ccle_gygi,randomforest,rna,ELACESTRANT (BRD:BRD-K00003295-300-01-9),0.388472,-0.040497,10,269,1584678
4,prot_ms_ccle_gygi,extratrees,rna+prot,ELACESTRANT (BRD:BRD-K00003295-300-01-9),0.387713,-0.092848,10,269,1584678
5,prot_ms_ccle_gygi,randomforest,rna,ELACESTRANT (BRD:BRD-K00003295-300-01-9),0.386894,-0.037176,10,269,17052356
6,prot_ms_ccle_gygi,extratrees,rna+prot,ELACESTRANT (BRD:BRD-K00003295-300-01-9),0.375876,-0.097148,10,269,19537
7,prot_ms_ccle_gygi,randomforest,rna,ELACESTRANT (BRD:BRD-K00003295-300-01-9),0.375129,-0.036741,10,269,19537
8,prot_ms_ccle_gygi,extratrees,rna,ELACESTRANT (BRD:BRD-K00003295-300-01-9),0.373807,-0.053196,10,269,19537
9,prot_ms_ccle_gygi,randomforest,rna+mut+prot,ELACESTRANT (BRD:BRD-K00003295-300-01-9),0.364855,-0.092642,10,269,19537



Worst observed rows (seed + drug + combo):


,arm,model,feature_set,compound_id,spearman,r2,n_folds,n_test_total,seed
0,prot_ms_ccle_gygi,extratrees,cnv+prot,ATRACTYLENOLIDE-I (BRD:BRD-K00003732-001-01-9),-0.243744,-0.139796,10,269,17052356
1,prot_ms_ccle_gygi,extratrees,cnv+mut,ATRACTYLENOLIDE-I (BRD:BRD-K00003732-001-01-9),-0.241287,-0.135354,10,269,1584678
2,prot_ms_ccle_gygi,extratrees,prot,AZD7594 (BRD:BRD-K00003100-001-01-9),-0.235786,-0.125510,10,269,17052356
3,prot_ms_ccle_gygi,randomforest,mut+prot,PEMIROLAST (BRD:BRD-K31731454-237-04-9),-0.230818,-0.110857,10,269,1584678
4,prot_ms_ccle_gygi,xgb_quick,rna+cnv,ASP-6537 (BRD:BRD-K82239282-001-02-9),-0.228638,-0.455304,10,269,19537
5,prot_ms_ccle_gygi,extratrees,cnv+mut,ATRACTYLENOLIDE-I (BRD:BRD-K00003732-001-01-9),-0.227942,-0.130700,10,269,19537
6,prot_ms_ccle_gygi,randomforest,cnv+mut,RETAGLIPTIN (BRD:BRD-K00003118-011-01-9),-0.223003,-0.133264,10,269,1584678
7,prot_ms_ccle_gygi,extratrees,cnv+mut+prot,RETAGLIPTIN (BRD:BRD-K00003118-011-01-9),-0.222271,-0.094082,10,269,1584678
8,prot_ms_ccle_gygi,ridge,prot,PEMIROLAST (BRD:BRD-K31731454-237-04-9),-0.221901,-1.800165,10,269,19537
9,prot_ms_ccle_gygi,ridge,prot,PEMIROLAST (BRD:BRD-K31731454-237-04-9),-0.221901,-1.800165,10,269,1584678


# 1. Overview and Experimental Design

This notebook represents the first quantitative evaluation stage of the multi-modal drug sensitivity prediction pipeline. Its central aim is to address a fundamental question: does the addition of proteomics data to a genomic backbone improve the prediction of cancer cell line drug sensitivity? Rather than presuming that one proteomics resource is inherently superior, the experiment evaluates four distinct proteomics arms in parallel: CCLE mass spectrometry (Gygi lab, `prot_ms_ccle_gygi`), CCLE reverse-phase protein array (`prot_rppa_ccle`), ProCan-DepMapSanger (`prot_procan_depmapSanger`), and a namespaced union of all three (`prot_combined_union`). This design ensures that the selection of a proteomics arm is guided by empirical evidence rather than prior assumption.

The drug response endpoint used in this notebook is log-fold-change (LFC) relative to DMSO, measured at a single fixed dose of 2.5 µM from the PRISM Repurposing primary screen. This constitutes a deliberately difficult prediction target. Unlike area-under-the-curve (AUC) metrics, which summarise signal across a full dose-response profile, single-dose LFC captures only one point on that curve and is therefore substantially noisier. Evaluating performance under this more challenging setting provides a useful lower bound that helps contextualise the results obtained from the AUC-based analyses.

The experimental structure includes 100 drugs selected solely on the basis of PRISM coverage, meaning the number of cell lines for which a measurement is available. Importantly, no selection is made according to predictability or known biological relevance. This is a crucial anti-leakage decision because selecting drugs on the basis of favourable modelling performance would artificially inflate the reported metrics and undermine the validity of the comparison. For each drug, a lineage-aware `GroupKFold` cross-validation strategy is applied, with folds split by cancer lineage rather than assigned randomly. Consequently, the model is always evaluated on cell lines from lineages absent during training, yielding a conservative and biologically credible estimate of generalisation. The full experiment is repeated across three independent random seeds in order to assess the stability of the findings.

The feature space is intentionally exhaustive. All 15 possible combinations of the four modalities, namely RNA, CNV, mutation, and proteomics, are evaluated for each drug, proteomics arm, model, and seed. This combinatorial structure makes it possible to conduct the Shapley-style marginal contribution analysis discussed later in the notebook.

# 2. Models and Preprocessing

Five model families are benchmarked: Ridge regression, ElasticNet, Extra Trees, Random Forest, and XGBoost. This selection is deliberate because linear models such as Ridge and ElasticNet, and tree-based ensemble methods such as Extra Trees, Random Forest, and XGBoost, embody fundamentally different inductive biases. A finding that remains consistent across all five model classes is therefore considerably more convincing than one that depends on a single modelling framework.

All preprocessing is carried out in a strictly leakage-safe manner. The imputer, scaler, and PCA are each fitted only on the training-fold cell lines and then applied to the corresponding validation fold. This procedure is enforced through the `FoldTransformer` class, which encapsulates the entire preprocessing pipeline and is instantiated independently for every fold. Each modality is reduced to a maximum of 200 principal components prior to concatenation, thereby controlling dimensionality while preserving the major axes of variation. For proteomics arms with substantial missingness, any columns that are entirely missing within the training fold are removed before fitting, ensuring that the imputer is never applied to a column containing no informative signal.

# 3. Overall Performance

The merged results file contains 90,000 rows, corresponding to the complete factorial design of 100 drugs, 4 proteomics arms, 5 models, 15 feature combinations, and 3 random seeds. Across this full experimental space, the global mean Spearman correlation is 0.024 and the global median is 0.018, while the mean R² is approximately -1.06. The strongly negative R² values are especially revealing. They indicate that many model configurations predict drug sensitivity less effectively than a trivial null model that always predicts the training-set mean for every cell line. This does not suggest an implementation error. Rather, it reflects the intrinsic difficulty of predicting single-dose LFC response.

A particularly important result is that seed variance is negligible. The best-performing seed, 1584678, achieves a mean Spearman of 0.0245, whereas the worst-performing seed, 19537, achieves 0.0241. The difference of 0.0003 is scientifically trivial. This stability is reassuring because it suggests that the observed results reflect genuine signal, or the genuine absence of signal, rather than favourable random partitions. It also indicates that the lineage-aware `GroupKFold` scheme is producing highly consistent cohort assignments across seeds.

# 4. Best and Worst Feature Combinations

The best-performing configuration across all three seeds is `prot_combined_union` with Extra Trees using RNA features alone, achieving a mean Spearman correlation of approximately 0.056. Two aspects of this result merit particular attention. First, the top-performing model relies on RNA alone rather than on RNA combined with proteomics. This suggests that, under the difficult conditions of LFC prediction, the broader cell line coverage provided by the union arm, which includes 679 lines with at least some proteomics data, is more beneficial than the additional feature dimensions offered by proteomics itself. In other words, the increased number of training samples appears to matter more than added feature richness in this setting. Second, the success of Extra Trees implies that tree ensembles are better suited than linear models to the high-variance, moderate-sample regime associated with this task.

At the opposite end of the spectrum, mutation-only features perform worst, producing consistently negative Spearman correlations, approximately -0.015 to -0.016, across all three seeds and multiple proteomics arms. A negative Spearman correlation indicates that the model is systematically reversing the true ranking of sensitivity, predicting resistant cell lines as sensitive and vice versa. This is a strong and reproducible finding. In this setting, binary mutation matrices appear to be anti-informative for LFC prediction. The likely explanation is that damaging mutations are sparse across cell lines, and when combined with the high noise level of single-dose LFC, mutation-based models are prone to learning spurious associations in the training data that fail to generalise and may even invert in the test data.

# 5. Proteomics Uplift Analysis

The uplift analysis examines, for each proteomics arm and model combination, what proportion of the 100 drugs benefit from adding proteomics to the genomic backbone (`rna+cnv+mut`), and by what magnitude. The results reveal a nuanced picture that varies substantially by both proteomics arm and model family.

The most favourable configuration is `prot_combined_union` with Ridge regression. In this case, proteomics improves 52% of drugs, defined by a Spearman delta greater than 0.01, with a mean delta of +0.011. This is the only setting in which proteomics benefits more than half of all drugs, making it the strongest positive signal observed in the entire LFC benchmark. The linear structure of Ridge regression is likely relevant here. The union arm contains high-dimensional data with structured missingness arising primarily from platform availability, and a regularised linear model appears better able to accommodate this pattern than tree-based ensembles, which may overfit to platform-driven missingness blocks.

At the other extreme, `prot_ms_ccle_gygi` with ElasticNet actively reduces performance, with a mean delta of -0.008 and a fraction helped of only 0.31. A plausible explanation is that the L1 component of ElasticNet drives aggressive feature sparsification, zeroing out protein variables whose signal-to-noise ratio falls below the regularisation threshold. In this setting, that sparsification does not appear sufficiently selective to preserve genuinely informative proteins, and therefore removes useful signal alongside noise.

A broader observation, which extends beyond any individual arm-model pairing, concerns the width of the interquartile range for uplift deltas, which spans roughly -0.03 to +0.04 for most arms. This indicates that proteomics strongly benefits some drugs while harming others. Consequently, a mean uplift close to zero should not be interpreted as neutrality. Rather, it reflects the averaging of large positive and negative drug-specific effects. This heterogeneity underscores why per-drug uplift analysis and drug-level stability measures are more informative than arm-level summary statistics on their own.

# 6. Backbone Lock Evidence: Coverage Versus Performance

One of the key deliverables of this notebook is evidence to support the proteomics backbone lock decision, which determines which arm or arms should be used in later modelling stages. The coverage-versus-performance table combines arm-level statistics, including PRISM cell line overlap, overall missingness, and feature count, with Ridge-model uplift metrics in order to compute a composite lock score.

Under the chosen lock-score formulation, which rewards overlap and uplift while penalising missingness, RPPA ranks first with a score of 0.104. However, this ranking is driven primarily by RPPA's complete data structure and its 505 overlapping cell lines rather than by strong proteomics signal. Its mean uplift delta is only +0.003, the weakest of all four arms. This reveals an important limitation of the heuristic: a complete but weakly informative proteomics arm can score artificially well because the missingness penalty dominates the formula. By contrast, the combined union arm yields the strongest uplift, +0.011, but is heavily penalised for its 58.9% overall missingness rate, much of which is structural and arises from platform absence rather than measurement failure at the individual protein level.

The most practical conclusion from the LFC evidence is therefore to treat coverage and performance as separate dimensions rather than collapsing them into a single score. `prot_procan_depmapSanger` emerges as the most suitable primary Track 1 arm because it offers a balanced combination of manageable missingness (34.9%), reasonable PRISM overlap (395 lines), and positive uplift. Meanwhile, `prot_combined_union` should be retained as the Track 2 missing-modality stress-test arm, where its high structural missingness becomes part of the experimental challenge rather than a simple disadvantage.

# 7. Modality Marginal Contributions (Shapley-Style Decomposition)

The modality marginal contribution analysis produces one of the most theoretically grounded findings in the notebook. For each modality, the average gain in Spearman correlation obtained by adding that modality to every possible subset that does not already contain it is computed, averaging across all drugs, proteomics arms, and seeds. This is formally equivalent to the Shapley value of each modality in a cooperative game in which the reward is predictive performance.

The resulting ranking is highly consistent across all five model families:

**RNA >> PROT ≈ CNV >> MUT**

RNA contributes approximately +0.012 to +0.024 mean Spearman, depending on the model family. Proteomics contributes around +0.005 to +0.007, which is smaller but still consistently positive. CNV contributes approximately +0.002 to +0.006, while mutation contributes negatively, around -0.002 to -0.008, across all models.

Several important implications follow. First, transcriptomic state is clearly the dominant predictor of drug sensitivity at the single-dose LFC level. This aligns with prior PRISM-related literature and with the broader biological understanding that transcriptional programmes encode differentiation state, proliferative behaviour, and pathway activity, all of which shape drug response. Second, proteomics provides a modest but consistently positive marginal gain. Regardless of which other modalities are already present, protein measurements improve predictive performance on average. This constitutes the principal justification for retaining proteomics in downstream stages of the project. Third, the negative contribution of mutation data suggests that binary mutation matrices introduce more noise than signal at this difficulty level. Their inclusion should therefore be conditional and perhaps restricted to drug classes where mutation-driven sensitivity is biologically well established, such as certain targeted therapies.

# 8. Top and Bottom Predictable Drugs

Across all three seeds, the most predictable drug is ELACESTRANT, a selective oestrogen receptor degrader (SERD), with a mean best-combination Spearman of 0.40 and a standard deviation of 0.010. This indicates both strong predictive performance and high stability. The best-performing feature set, `rna+prot` using `prot_ms_ccle_gygi` with Random Forest or Extra Trees, is also biologically interpretable. Elacestrant sensitivity is largely determined by oestrogen receptor expression status, which is captured at both transcript and protein levels. The fact that `rna+prot` outperforms `rna` alone suggests that protein-level features related to the ER pathway, such as receptor abundance or co-activator expression, provide useful signal beyond transcript counts alone.

Other high-performing drugs include ATUVECICLIB, a CDK9 inhibitor, with Spearman around 0.364, ERIBULIN, a microtubule inhibitor, with Spearman around 0.344, and IACS-10759, an OXPHOS inhibitor, with Spearman around 0.302. Notably, ATUVECICLIB and ERIBULIN show zero standard deviation across seeds in the top-drugs table. This should be interpreted cautiously. Zero variance across three seeds does not necessarily imply perfect seed invariance. It may instead arise when a drug has few evaluated folds and the fold-mean Spearman becomes deterministic given the fixed data loading order. These drugs are therefore promising candidates for further analysis, but not yet definitive examples of true seed-invariant predictability without additional fold-level variance checks.

At the lower end, ATRACTYLENOLIDE-I, a natural sesquiterpene lactone, achieves a best-combination Spearman of only 0.094, with some worst-observed rows reaching -0.244. Its broad and relatively non-specific mechanism of action, involving multiple targets without a clearly dominant pathway, likely makes it difficult for genomic or proteomic features to stratify sensitivity reliably across cell lines. Similarly, DOTMP and KR-33494 exhibit extremely negative R² values, -7.9 and -1.5 respectively, suggesting that the models are fitting noise rather than meaningful biological structure.

# 9. Stable Drug Candidates for Interpretability Analysis

The drug stability report identifies 20 drugs that satisfy both a minimum performance threshold, namely minimum Spearman greater than 0.05 across all seeds, and a stability threshold, namely coefficient of variation below 0.30. These drugs represent the strongest candidates for downstream interpretability analysis in Step 14 of the project plan, because their consistent predictability across seeds and proteomics arms provides greater confidence that attribution methods such as SHAP or integrated gradients will reflect real biological patterns rather than artefacts of random initialisation.

However, the presence of zero coefficient of variation for several entries in this list reflects the same deterministic fold-mean phenomenon noted previously. Before selecting these drugs for final case-study analysis, it would be prudent to examine fold-level stability more closely. Drugs that recur in the stable-candidate list across multiple proteomics arms are especially compelling. For example, ATUVECICLIB appears for both `prot_ms_ccle_gygi` and `prot_procan_depmapSanger`, and IACS-10759 shows similar cross-arm consistency. Such replication across proteomics arms provides an additional layer of validation by indicating that the predictive signal is not tied to a single proteomics source.

# 10. Summary

The LFC benchmark demonstrates that genomic prediction of single-dose drug sensitivity is possible, but only at modest levels of performance. The best drug-specific models achieve Spearman correlations of approximately 0.3 to 0.4, while the global mean remains close to 0.024. Proteomics contributes a consistently positive marginal gain across all model families, which supports its inclusion in later stages of the modelling pipeline. However, the size of this gain is small relative to RNA, and it is sensitive to both the choice of proteomics arm and the regularisation characteristics of the model. By contrast, the mutation modality appears consistently harmful in this setting and should be incorporated with caution. Taken together, and considered alongside the AUC analyses, these findings provide the empirical basis for the backbone lock decision that will shape all downstream experiments.



In [32]:
# Per-drug uplift distribution 

def compute_per_drug_uplift(drug_means_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each (seed, arm, model, compound_id): delta Spearman when adding PROT
    to the rna+cnv+mut backbone. Only rows where both feature sets exist are kept.
    """
    base = drug_means_df[drug_means_df["feature_set"] == "rna+cnv+mut"][
        ["seed", "arm", "model", "compound_id", "spearman", "r2"]
    ].rename(columns={"spearman": "sp_base", "r2": "r2_base"})

    full = drug_means_df[drug_means_df["feature_set"] == "rna+cnv+mut+prot"][
        ["seed", "arm", "model", "compound_id", "spearman", "r2"]
    ].rename(columns={"spearman": "sp_full", "r2": "r2_full"})

    joined = full.merge(base, on=["seed", "arm", "model", "compound_id"], how="inner")
    joined["delta_spearman"] = joined["sp_full"] - joined["sp_base"]
    joined["delta_r2"] = joined["r2_full"] - joined["r2_base"]
    # A delta > 0.01 is treated as a meaningful improvement (not noise)
    joined["prot_helped"] = joined["delta_spearman"] > 0.01
    return joined

# Load the merged per-drug aggregated file produced above
per_drug_uplift = compute_per_drug_uplift(merged)

uplift_dist_path = OUT_REPORTS / f"per_drug_uplift_distribution_{TARGET}.csv"
per_drug_uplift.to_csv(uplift_dist_path, index=False)
print("Wrote:", uplift_dist_path, per_drug_uplift.shape)

# Summary: per (arm, model) — fraction of drugs helped, quartiles of delta
uplift_summary = (
    per_drug_uplift
    .groupby(["arm", "model"], as_index=False)
    .agg(
        n_drugs=("compound_id", "nunique"),
        frac_helped=("prot_helped", "mean"),
        mean_delta_sp=("delta_spearman", "mean"),
        median_delta_sp=("delta_spearman", "median"),
        p25_delta=("delta_spearman", lambda x: float(x.quantile(0.25))),
        p75_delta=("delta_spearman", lambda x: float(x.quantile(0.75))),
        mean_sp_base=("sp_base", "mean"),
        mean_sp_full=("sp_full", "mean"),
    )
    .sort_values("mean_delta_sp", ascending=False)
)

uplift_summary_path = OUT_REPORTS / f"uplift_summary_by_arm_model_{TARGET}.csv"
uplift_summary.to_csv(uplift_summary_path, index=False)
print("Wrote:", uplift_summary_path)

print("\nProteomics uplift summary (frac_helped = fraction of drugs where PROT delta > 0.01):")
display(uplift_summary)

# Modality marginal contributions (Shapley-style)

def modality_marginal_contributions(
    drug_means_df: pd.DataFrame,
    model_name: str = "ridge",
) -> pd.DataFrame:
    """
    For each modality M, average Spearman gain from adding M to every subset
    that does not already contain M. Averaged across (drug, arm, seed).
    This is the Shapley value of each modality over the 4-modality game.
    """
    MODS = ["rna", "cnv", "mut", "prot"]
    df = drug_means_df[drug_means_df["model"] == model_name].copy()
    df["fs_set"] = df["feature_set"].apply(lambda s: frozenset(s.split("+")))

    rows = []
    for mod in MODS:
        other_mods = [m for m in MODS if m != mod]
        deltas = []

        for r in range(0, len(MODS)):  # subset sizes 0..3 (subsets not containing mod)
            for without_tuple in _combinations(other_mods, r):
                without_set = frozenset(without_tuple)
                with_set = without_set | {mod}

                sub_with = df[df["fs_set"] == with_set][
                    ["seed", "arm", "compound_id", "spearman"]
                ].rename(columns={"spearman": "sp_with"})

                if len(without_tuple) > 0:
                    sub_without = df[df["fs_set"] == without_set][
                        ["seed", "arm", "compound_id", "spearman"]
                    ].rename(columns={"spearman": "sp_without"})

                    joined = sub_with.merge(
                        sub_without,
                        on=["seed", "arm", "compound_id"],
                        how="inner",
                    )
                    if joined.shape[0] > 0:
                        vals = (joined["sp_with"] - joined["sp_without"]).to_numpy(dtype=float)
                        vals = vals[np.isfinite(vals)]
                        deltas.extend(vals.tolist())
                else:
                    # Adding mod to the empty set: raw Spearman of mod-only
                    if sub_with.shape[0] > 0:
                        vals = sub_with["sp_with"].to_numpy(dtype=float)
                        vals = vals[np.isfinite(vals)]
                        deltas.extend(vals.tolist())

        arr = np.asarray(deltas, dtype=float)
        arr = arr[np.isfinite(arr)]

        rows.append({
            "modality": mod,
            "model": model_name,
            "mean_marginal_contribution": float(np.nanmean(arr)) if arr.size else np.nan,
            "median_marginal_contribution": float(np.nanmedian(arr)) if arr.size else np.nan,
            "std_marginal_contribution": float(np.nanstd(arr)) if arr.size else np.nan,
            "n_comparisons": int(arr.size),
        })

    return pd.DataFrame(rows).sort_values("mean_marginal_contribution", ascending=False)

shapley_rows = []
for m_name in merged["model"].unique():
    shapley_rows.append(modality_marginal_contributions(merged, model_name=m_name))

shapley_df = pd.concat(shapley_rows, ignore_index=True).sort_values(
    ["model", "mean_marginal_contribution"], ascending=[True, False]
)

shapley_path = OUT_REPORTS / f"modality_marginal_contributions_{TARGET}.csv"
shapley_df.to_csv(shapley_path, index=False)
print("Wrote:", shapley_path)

print("\nModality marginal contributions (Shapley-style, per model):")
display(shapley_df)

# Coverage vs performance join 

def coverage_vs_performance(
    coverage_df: pd.DataFrame,
    uplift_summary_df: pd.DataFrame,
    model_name: str = "ridge",
) -> pd.DataFrame:
    """
    Joins arm-level coverage stats with performance uplift.
    Directly supports the backbone lock decision.
    """
    cov = coverage_df[[
        "arm", "n_overlap_with_prism_lfc_cells", "overall_missing_pct",
        "n_features", "n_cells_with_any_prot",
    ]].copy()

    perf = uplift_summary_df[uplift_summary_df["model"] == model_name][[
        "arm", "frac_helped", "mean_delta_sp", "median_delta_sp",
        "mean_sp_base", "mean_sp_full",
    ]].copy()

    joined = cov.merge(perf, on="arm", how="left")

    # Composite lock score: same formula as the existing heuristic but now explicit
    joined["lock_score"] = (
        joined["mean_delta_sp"].fillna(-1e9)
        + 0.0002 * joined["n_overlap_with_prism_lfc_cells"].astype(float)
        - 0.001  * joined["overall_missing_pct"].fillna(100.0)
    )
    return joined.sort_values("lock_score", ascending=False).reset_index(drop=True)

cov_perf = coverage_vs_performance(prot_backbone_coverage, uplift_summary)
cov_perf_path = OUT_REPORTS / f"prot_arm_coverage_vs_performance_{TARGET}.csv"
cov_perf.to_csv(cov_perf_path, index=False)
print("Wrote:", cov_perf_path)

print("\nCoverage vs performance (backbone lock evidence):")
display(cov_perf)

# Drug-level stability across seeds

def drug_stability_report(
    merged_df: pd.DataFrame,
    model_name: str = "ridge",
    feature_set: str = "rna+cnv+mut+prot",
) -> pd.DataFrame:
    """
    Per (arm, compound_id): mean/std Spearman across seeds.
    Stable + positive drugs are prime candidates for interpretability case studies.
    """
    sub = merged_df[
        (merged_df["model"] == model_name) &
        (merged_df["feature_set"] == feature_set)
    ].copy()

    if sub.shape[0] == 0:
        print(f"[WARN] drug_stability_report: no rows for model={model_name}, feature_set={feature_set}")
        return pd.DataFrame()

    stability = (
        sub.groupby(["arm", "compound_id"], as_index=False)
        .agg(
            n_seeds=("seed", "nunique"),
            mean_spearman=("spearman", "mean"),
            std_spearman=("spearman", "std"),
            min_spearman=("spearman", "min"),
            max_spearman=("spearman", "max"),
            mean_r2=("r2", "mean"),
        )
    )

    # CV = std / |mean|; low CV means consistent across seeds
    stability["cv_spearman"] = stability["std_spearman"] / (stability["mean_spearman"].abs() + 1e-9)

    # Flag drugs that are positive AND stable across ALL seeds seen
    stability["is_stable_positive"] = (
        (stability["min_spearman"] > 0.05) &   # positive in every seed
        (stability["cv_spearman"] < 0.30)       # low variance across seeds
    )

    return stability.sort_values(
        ["is_stable_positive", "mean_spearman"],
        ascending=[False, False],
    ).reset_index(drop=True)

stability_df = drug_stability_report(merged)

stability_path = OUT_REPORTS / f"drug_stability_across_seeds_{TARGET}.csv"
stability_df.to_csv(stability_path, index=False)
print("Wrote:", stability_path)

stable_candidates = stability_df[stability_df["is_stable_positive"]].head(20)
print(f"\nStable + positive drugs (candidates for Step 14 case studies): {stable_candidates.shape[0]} found")
display(stable_candidates)

Wrote: artifacts\reports\notebook 3a\per_drug_uplift_distribution_lfc.csv (6000, 11)
Wrote: artifacts\reports\notebook 3a\uplift_summary_by_arm_model_lfc.csv

Proteomics uplift summary (frac_helped = fraction of drugs where PROT delta > 0.01):


,arm,model,n_drugs,frac_helped,mean_delta_sp,median_delta_sp,p25_delta,p75_delta,mean_sp_base,mean_sp_full
3,prot_combined_union,ridge,100,0.520000,0.011356,0.013240,-0.026827,0.044592,0.016060,0.027416
14,prot_procan_depmapSanger,xgb_quick,100,0.460000,0.007255,0.004960,-0.027772,0.043096,0.022832,0.030087
0,prot_combined_union,elasticnet,100,0.390000,0.004705,-0.000955,-0.027219,0.034706,0.028304,0.033009
13,prot_procan_depmapSanger,ridge,100,0.380000,0.004676,-0.000877,-0.022428,0.027120,0.032364,0.037039
12,prot_procan_depmapSanger,randomforest,100,0.416667,0.004448,0.002799,-0.016723,0.026888,0.030464,0.034912
10,prot_procan_depmapSanger,elasticnet,100,0.380000,0.004361,-0.002026,-0.019534,0.027908,0.030807,0.035169
6,prot_ms_ccle_gygi,extratrees,100,0.466667,0.004237,0.006374,-0.029974,0.032190,0.022302,0.026539
9,prot_ms_ccle_gygi,xgb_quick,100,0.470000,0.003924,0.005192,-0.033996,0.044322,0.018034,0.021958
11,prot_procan_depmapSanger,extratrees,100,0.430000,0.003100,0.002055,-0.024527,0.027985,0.038932,0.042032
18,prot_rppa_ccle,ridge,100,0.270000,0.002817,0.002189,-0.005170,0.010844,0.014998,0.017815


Wrote: artifacts\reports\notebook 3a\modality_marginal_contributions_lfc.csv

Modality marginal contributions (Shapley-style, per model):


,modality,model,mean_marginal_contribution,median_marginal_contribution,std_marginal_contribution,n_comparisons
0,rna,elasticnet,0.012808,0.008841,0.073638,9600
1,prot,elasticnet,0.005086,-0.000187,0.061917,9600
2,cnv,elasticnet,0.004779,0.005014,0.069935,9600
3,mut,elasticnet,-0.003827,-0.004431,0.049534,9600
4,rna,extratrees,0.024498,0.018638,0.069204,9600
5,prot,extratrees,0.006583,0.002040,0.058023,9600
6,cnv,extratrees,0.001923,-0.000595,0.061152,9600
7,mut,extratrees,-0.001697,-0.001059,0.039167,9600
8,rna,randomforest,0.023039,0.018155,0.063115,9600
9,prot,randomforest,0.006122,0.003398,0.053166,9600


Wrote: artifacts\reports\notebook 3a\prot_arm_coverage_vs_performance_lfc.csv

Coverage vs performance (backbone lock evidence):


,arm,n_overlap_with_prism_lfc_cells,overall_missing_pct,n_features,n_cells_with_any_prot,frac_helped,mean_delta_sp,median_delta_sp,mean_sp_base,mean_sp_full,lock_score
0,prot_rppa_ccle,505,0.000000,144,612,0.27,0.002817,0.002189,0.014998,0.017815,0.103817
1,prot_combined_union,538,58.871158,18751,679,0.52,0.011356,0.013240,0.016060,0.027416,0.060084
2,prot_procan_depmapSanger,395,34.854124,7906,485,0.38,0.004676,-0.000877,0.032364,0.037039,0.048822
3,prot_ms_ccle_gygi,277,25.025104,11780,304,0.39,0.002081,0.000733,0.031549,0.033630,0.032456


Wrote: artifacts\reports\notebook 3a\drug_stability_across_seeds_lfc.csv

Stable + positive drugs (candidates for Step 14 case studies): 20 found


,arm,compound_id,n_seeds,mean_spearman,std_spearman,min_spearman,max_spearman,mean_r2,cv_spearman,is_stable_positive
0,prot_ms_ccle_gygi,ATUVECICLIB (BRD:BRD-K00003144-001-01-9),3,0.281018,0.0,0.281018,0.281018,-0.360678,0.0,True
1,prot_procan_depmapSanger,IACS-10759 (BRD:BRD-K00003105-003-01-9),3,0.280789,0.0,0.280789,0.280789,-0.309153,0.0,True
2,prot_ms_ccle_gygi,ELACESTRANT (BRD:BRD-K00003295-300-01-9),3,0.279928,0.0,0.279928,0.279928,-1.412575,0.0,True
3,prot_ms_ccle_gygi,ERIBULIN (BRD:BRD-K00003156-066-01-9),3,0.276819,0.0,0.276819,0.276819,-0.270120,0.0,True
4,prot_procan_depmapSanger,ATUVECICLIB (BRD:BRD-K00003144-001-01-9),3,0.234689,0.0,0.234689,0.234689,-0.245944,0.0,True
5,prot_procan_depmapSanger,CEP-40783 (BRD:BRD-K00003396-001-01-9),3,0.230984,0.0,0.230984,0.230984,-0.361896,0.0,True
6,prot_procan_depmapSanger,OMTRIPTOLIDE (BRD:BRD-K00003212-001-01-9),3,0.227965,0.0,0.227965,0.227965,-0.422386,0.0,True
7,prot_ms_ccle_gygi,IACS-10759 (BRD:BRD-K00003105-003-01-9),3,0.218764,0.0,0.218764,0.218764,-1.938507,0.0,True
8,prot_combined_union,AZ-628 (BRD:BRD-K05804044-001-18-5),3,0.215932,0.0,0.215932,0.215932,-1.149947,0.0,True
9,prot_ms_ccle_gygi,C188-9 (BRD:BRD-K00003114-001-01-9),3,0.214070,0.0,0.214070,0.214070,-0.382014,0.0,True


# 1. Purpose of the EDA Extensions

The core benchmark loop described in the main results section generates a large matrix of per-drug, per-fold Spearman correlations. However, these raw values do not in themselves answer the scientific questions that matter for backbone selection and downstream modelling. The EDA extensions convert those raw outputs into four focused analyses, each designed to address a specific question.

The per-drug uplift distribution examines whether proteomics provides a consistent benefit or whether its value is confined to a minority of drugs. The Shapley-style modality decomposition evaluates how much each individual omics layer contributes to predictive performance on average when assessed fairly across all possible feature combinations. The coverage-versus-performance join asks whether the proteomics arms with the strongest predictive signal are also those with the best cell line coverage, or whether a trade-off exists between these two properties. Finally, the drug stability report identifies which drugs remain reliably predictable across random seeds and are therefore appropriate candidates for interpretability analysis. Collectively, these four analyses form the evidential basis for the proteomics backbone lock decision.

# 2. Per-Drug Uplift Distribution

## 2.1 What Is Being Measured

The uplift analysis computes, for each drug independently, the difference in Spearman correlation obtained when proteomics is added to the `rna+cnv+mut` backbone compared with using that backbone alone. This per-drug delta is substantially more informative than an arm-level mean. A mean uplift close to zero could indicate either a uniformly neutral contribution from proteomics or a bimodal pattern in which proteomics strongly improves prediction for some drugs while substantially worsening it for others. A drug is classified as *helped* if its delta exceeds 0.01 Spearman, a threshold chosen to distinguish a practically meaningful improvement from numerical noise at this performance level.

## 2.2 Results and Interpretation

The results show that no proteomics arm or model combination delivers universal benefit across all 100 drugs. The best-performing combination is `prot_combined_union` with Ridge regression, where proteomics improves 52% of drugs. This makes it the only configuration in the entire LFC experiment for which a majority of drugs benefit from the addition of proteomics. Its mean delta of +0.011 is also the highest observed, while its positive median delta of +0.013 indicates that the effect is not driven merely by a small number of strong outliers.

By contrast, `prot_ms_ccle_gygi` with ElasticNet produces a mean delta of -0.008, implying that this combination degrades performance on average. A likely explanation is that the L1 regularisation component of ElasticNet suppresses protein features whose marginal signal falls below the regularisation threshold. In this setting, that sparsification does not appear sufficiently selective, causing informative protein variables to be discarded alongside noisy ones and leading to a net reduction in predictive quality.

A recurring pattern throughout the uplift table is the consistently wide interquartile range, spanning approximately -0.03 to +0.04 for most arm-model combinations. This is an important finding because it indicates that even combinations with a positive mean delta still include a substantial proportion of drugs for which proteomics is detrimental. The appropriate interpretation is therefore not that proteomics is globally beneficial or globally harmful, but that its usefulness is highly drug-specific. This likely depends on whether a drug's mechanism of action is better captured at the protein level than at the transcript level. Drugs whose sensitivity depends on post-translational regulation, protein stability, or phosphorylation state would be expected to benefit more from proteomics, whereas drugs whose response is primarily governed by broad transcriptional programme identity would be expected to benefit less.

A secondary but useful observation is that the `frac_helped` metric and the `mean_delta_sp` metric do not always rank arm-model combinations in the same order. For example, `prot_ms_ccle_gygi` with XGBoost has a relatively high `frac_helped` of 0.47 but only a modest mean delta of +0.004, whereas `prot_combined_union` with Ridge has the highest mean delta but a `frac_helped` of 0.52. This difference arises because `frac_helped` measures how many drugs cross the 0.01 threshold, while mean delta reflects the average magnitude of benefit. For backbone selection, `mean_delta_sp` is the more practically relevant metric because it better captures the expected improvement for a randomly chosen drug from the coverage-selected set.

# 3. Modality Marginal Contributions (Shapley-Style Decomposition)

## 3.1 Methodological Basis

The Shapley value framework, originally developed in cooperative game theory, provides a principled method for attributing a total reward among contributing players. In this context, the players are the four omics modalities: RNA, CNV, mutation, and proteomics. The reward is predictive performance, measured using Spearman correlation for the best-performing model configuration. For each modality, the Shapley value is computed by averaging the marginal gain obtained when that modality is added to every possible subset of the remaining modalities, with equal weighting across subset sizes.

This averaging is essential because it means that the value of proteomics, for example, is not measured only when added to the full `rna+cnv+mut` backbone, but also when added to RNA alone, CNV alone, the empty set, and every other possible combination. The result is a single value per modality that reflects its average contribution across all possible contexts, making it robust to whichever specific feature combination happens to perform best in a particular setting.

With 100 drugs, 4 proteomics arms, and 3 random seeds, each modality's Shapley value is estimated from 9,600 individual comparisons per model family, providing a well-powered estimate with low variance.

## 3.2 Results and Interpretation

The modality ranking is remarkably consistent across all five model families, and this consistency is itself one of the central findings. RNA contributes between +0.012 and +0.025 mean Spearman, depending on the model family, making it the dominant modality by a factor of approximately three to four compared with any other feature group. This is consistent with the existing literature on drug sensitivity prediction, in which transcriptomic features have repeatedly outperformed other omics layers in PRISM-scale tasks. The transcriptome captures cellular differentiation state, pathway activation, and proliferative capacity, all of which are major determinants of broad-spectrum drug sensitivity.

Proteomics contributes between +0.005 and +0.007 across all model families, placing it consistently in second position. Importantly, this contribution is positive for every model family without exception. This means that the signal contained in protein measurements is genuine and not an artefact of a particular model's inductive bias. The fact that its contribution remains smaller than that of RNA in the LFC setting reflects the difficulty of the task. At single dose, the underlying noise level is high enough that the additional value of proteomics is partially obscured, even though it remains consistently present.

CNV contributes approximately +0.002 to +0.006 and occupies third position, providing a modest but positive signal. This is biologically plausible, as copy number alterations influence oncogene and tumour suppressor dosage and therefore overlap partly with transcriptomic information while still offering structural genomic context not fully captured by RNA alone.

The mutation modality yields the most practically important result. Its marginal contribution is negative across all five model families, ranging from -0.002 to -0.008. A negative Shapley value indicates that, on average, adding mutation features to any existing combination makes prediction slightly worse. This conclusion is robust, being supported by 9,600 comparisons per model family. A likely explanation is that binary damaging mutation matrices are extremely sparse, with most cell lines carrying only a small number of damaging events in any given gene. This sparsity leads to a low signal-to-noise ratio, preventing mutation features from contributing positively at the whole-dataset level. Certain individual drugs with known mutation-driven sensitivity, such as BRAF inhibitors in BRAF-mutant lines, may still benefit from mutation information, but these cases appear to be the exception rather than the rule within the 100-drug coverage-selected set.

The overall ranking,

**RNA >> PROT ≈ CNV >> MUT**

should be understood as specific to the LFC prediction task under this level of noise. As the AUC results indicate, the relative importance of modalities shifts when a less noisy and more informative drug-response endpoint is used.

# 4. Coverage vs Performance: Backbone Lock Evidence

## 4.1 The Coverage-Performance Trade-off

The coverage-versus-performance table serves as the main decision-support output for the proteomics backbone lock. It combines three dimensions of arm quality: PRISM cell line overlap, overall missingness, and Ridge-model uplift metrics. PRISM overlap reflects how many cell lines contain both PRISM measurements and proteomics data. Missingness reflects the proportion of protein-cell line entries that are absent. Uplift metrics reflect the extent to which proteomics improves predictive performance when added to the backbone. A composite lock score is then computed that rewards overlap and uplift while penalising missingness.

## 4.2 Results and the Limitation of the Heuristic

Under the composite lock score, RPPA ranks first with a score of 0.104, driven by its zero missingness and the highest PRISM overlap of 505 cell lines. ProCan-DepMapSanger ranks second with 0.049, followed by `ms_ccle_gygi` with 0.032. However, closer inspection of the performance columns shows that RPPA's strong lock score is driven primarily by its coverage characteristics rather than its predictive usefulness. Its mean uplift delta of +0.003 and `frac_helped` value of 0.27 are the weakest among all four arms. In other words, RPPA provides the most complete data structure but the least informative proteomics signal for LFC prediction.

This mismatch highlights a familiar limitation of composite scoring heuristics. When the penalty on missingness is large relative to the reward assigned to predictive uplift, an arm with zero missingness will tend to score highly even if its actual predictive contribution is weak. In the present formula, the 0.001 missingness coefficient appears to reward completeness too strongly for the LFC setting, where the absolute uplift values are small, typically between 0.002 and 0.011 Spearman, and can therefore be easily dominated by the coverage term.

The most appropriate interpretation is therefore to treat coverage and performance as separate dimensions rather than collapsing them into a single score. Among the arms that provide meaningful positive uplift, `prot_procan_depmapSanger` offers the best balance. Its missingness rate of 34.9% is manageable, its 395 overlapping cell lines provide sufficient scale for Track 1 clean ablations, and its mean uplift of +0.005 with a `frac_helped` of 0.38 indicates genuine positive signal even under the difficult LFC setting. The combined union arm is best retained not as the main modelling arm but as the deliberate Track 2 missing-modality stress test. In that role, its high structural missingness becomes an advantage rather than a drawback because it reproduces the realistic scenario of platform-driven partial data availability that the modality-dropout training strategy is intended to address.

# 5. Drug Stability Across Seeds

## 5.1 Stability Criteria and Their Rationale

The drug stability report identifies drugs that remain reliably predictable regardless of which random seed is used for cross-validation. A drug is classified as *stably positive* if it satisfies two criteria simultaneously. First, its minimum Spearman correlation across all three seeds must exceed 0.05, ensuring that its positive predictability is not the result of a favourable single-seed outcome. Second, its coefficient of variation, defined as the standard deviation divided by the absolute mean, must be below 0.30, ensuring that performance does not fluctuate substantially across seeds.

These thresholds are intentionally conservative. A drug that only narrowly satisfies both criteria over three seeds may not remain stable under a fourth seed, whereas a drug with zero coefficient of variation and a mean Spearman comfortably above 0.05 provides a much stronger case for reproducible predictability.

Twenty drugs satisfy both conditions in the LFC experiment, spanning multiple proteomics arms and a range of drug classes. The leading examples, including ATUVECICLIB (CDK9 inhibitor), IACS-10759 (OXPHOS inhibitor), ELACESTRANT (SERD), and ERIBULIN (microtubule inhibitor), all achieve mean Spearman values above 0.27 on the `rna+cnv+mut+prot` feature set. These values place them well above the global mean of 0.024.

## 5.2 The Zero Coefficient of Variation Anomaly

A striking feature of this table is that every entry shows a standard deviation of exactly 0.0 and a coefficient of variation of exactly 0.0. This implies that, for each of these drugs, the cross-validation pipeline produced identical fold-mean Spearman values across all three seeds. Such perfect agreement requires careful interpretation, because true seed invariance of this kind would be unusual.

A more plausible explanation is that these drugs have a relatively small number of valid folds. The minimum fold-size criteria, requiring at least 10 test samples and 30 training samples, reduce the number of effective folds for some drugs. Under these circumstances, the fold-mean Spearman may be driven largely by which cell lines are present in the dataset rather than by seed-specific variation. In that case, the seed influences PCA initialisation and model stochasticity, but those factors contribute very little compared with the dominant underlying biological signal.

This interpretation suggests that the zero-coefficient-of-variation drugs are genuinely stable in the sense that their predictability is determined by the data rather than by random seed effects. In fact, this would represent a particularly strong form of stability. Nevertheless, before selecting these drugs for Step 14 case-study analysis, it would be prudent to verify this interpretation by inspecting fold-level variance directly. If individual fold Spearman values also show minimal variance across seeds, then the zero-coefficient-of-variation result is genuine. If substantial fold-level variation exists but cancels out in the mean, then the apparent stability may be at least partly artefactual.

## 5.3 Cross-Arm Replication as an Orthogonal Validation

Several drugs appear in the stable-candidate list for more than one proteomics arm. ATUVECICLIB appears for both `prot_ms_ccle_gygi` and `prot_procan_depmapSanger`. IACS-10759 appears for the same two arms. ASTX660 appears for both `prot_combined_union` and `prot_procan_depmapSanger`, and SAPACITABINE also appears for both `prot_combined_union` and `prot_procan_depmapSanger`.

Cross-arm replication constitutes a particularly strong form of validation because the different proteomics arms measure related but not identical biological quantities using different experimental platforms. The `ms_ccle_gygi` arm is based on mass spectrometry, whereas ProCan uses tandem mass tag-based deep proteomics. Their cell line cohorts also overlap only partially. A drug that remains consistently predictable across both platforms is therefore unlikely to owe its performance to platform-specific noise and instead represents a strong candidate for downstream interpretability analysis.

# 6. Summary

Taken together, the four EDA extensions provide a multi-dimensional characterisation of proteomics utility for LFC drug sensitivity prediction. The uplift distribution shows that proteomics benefit is heterogeneous and drug-specific, with the combined union arm under Ridge regression providing the most consistent positive effect. The Shapley decomposition establishes a clear and reproducible modality hierarchy in which RNA dominates, proteomics is consistently second, CNV is third, and mutation is reliably detrimental across all five model families. The coverage-versus-performance analysis reveals a meaningful trade-off between completeness and informativeness that cannot be resolved adequately by the composite lock score alone, thereby motivating a two-arm strategy in which ProCan serves as the primary Track 1 arm and the combined union arm serves as the Track 2 stress test. Finally, the stability analysis identifies twenty candidate drugs for Step 14 interpretability analysis, with cross-arm replicated cases representing the strongest candidates.

These findings collectively provide the empirical foundation for the backbone lock decision and establish the basis for the subsequent AUC analysis, which uses a more informative drug-response endpoint and is expected to reveal larger and more clearly differentiated proteomics contributions.

In [33]:
# Ranked feature-combination table with explicit proteomics source

def add_protein_source_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["uses_prot"] = out["feature_set"].astype(str).str.contains(r"(?:^|\+)prot(?:\+|$)", regex=True)    
    out["protein_dataset"] = np.where(out["uses_prot"], out["arm"], "none")
    return out

rank_input = add_protein_source_columns(merged)

# Rank exact evaluated combinations: arm + model + feature_set
feature_combo_ranking = (
    rank_input
    .groupby(["arm", "protein_dataset", "uses_prot", "model", "feature_set"], as_index=False)
    .agg(
        n_rows=("compound_id", "size"),
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", "mean"),
        median_spearman=("spearman", "median"),
        std_spearman=("spearman", "std"),
        mean_r2=("r2", "mean"),
        median_r2=("r2", "median"),
    )
    .sort_values(
        ["mean_spearman", "median_spearman", "mean_r2", "n_drugs"],
        ascending=[False, False, False, False]
    )
    .reset_index(drop=True)
)

feature_combo_ranking.insert(0, "rank", np.arange(1, len(feature_combo_ranking) + 1))

ranking_path = OUT_REPORTS / f"ranked_feature_combinations_{TARGET}.csv"
feature_combo_ranking.to_csv(ranking_path, index=False)
print("Wrote:", ranking_path)

print("\nTop ranked evaluated combinations:")
display(feature_combo_ranking.head(25))

Wrote: artifacts\reports\notebook 3a\ranked_feature_combinations_lfc.csv

Top ranked evaluated combinations:


,rank,arm,protein_dataset,uses_prot,model,feature_set,n_rows,n_drugs,mean_spearman,median_spearman,std_spearman,mean_r2,median_r2
0,1,prot_combined_union,none,False,extratrees,rna,300,100,0.055627,0.046388,0.082950,-0.046240,-0.038714
1,2,prot_rppa_ccle,none,False,extratrees,rna,300,100,0.054722,0.043008,0.086971,-0.047084,-0.043808
2,3,prot_combined_union,none,False,extratrees,rna+mut,300,100,0.053064,0.040117,0.081049,-0.046236,-0.038914
3,4,prot_combined_union,none,False,randomforest,rna,300,100,0.050670,0.035487,0.080705,-0.046380,-0.040255
4,5,prot_rppa_ccle,none,False,randomforest,rna,300,100,0.050144,0.037655,0.083694,-0.046250,-0.042121
5,6,prot_rppa_ccle,none,False,extratrees,rna+mut,300,100,0.049030,0.039806,0.086935,-0.047550,-0.044943
6,7,prot_combined_union,none,False,extratrees,rna+cnv+mut,300,100,0.049021,0.035615,0.075391,-0.047431,-0.041653
7,8,prot_procan_depmapSanger,prot_procan_depmapSanger,True,extratrees,rna+mut+prot,300,100,0.048999,0.034323,0.087473,-0.063778,-0.053913
8,9,prot_combined_union,none,False,extratrees,rna+cnv,300,100,0.048617,0.038521,0.075326,-0.047677,-0.041420
9,10,prot_procan_depmapSanger,prot_procan_depmapSanger,True,randomforest,rna+prot,300,100,0.047387,0.026880,0.088087,-0.064067,-0.053488
